In [2]:
!pip install -q roboflow pyyaml

In [4]:

from google.colab import drive, userdata
from roboflow import Roboflow
from pathlib import Path

drive.mount('/content/drive')

ROBOFLOW_KEY = userdata.get("ROBFLOW_KEY")

rf = Roboflow(api_key=ROBOFLOW_KEY)

BASE_DIR = Path(
    "/content/drive/MyDrive/Hazard_Detection/"
    "raw_datasets/obstacle"
)

SOURCE_DIR = BASE_DIR / "source_datasets"


Mounted at /content/drive


In [6]:

import shutil

DATASET_DIR = SOURCE_DIR / "bollards"

# Prevent accidental overwrite
if DATASET_DIR.exists():
    if any(DATASET_DIR.iterdir()):
        raise FileExistsError(
            f"Destination is not empty: {DATASET_DIR}"
        )
    else:
        DATASET_DIR.rmdir()

DATASET_DIR.mkdir(parents=True, exist_ok=True)

print("Connecting to Roboflow...")

project = (
    rf.workspace("project-60htx")
    .project("bollard-v2gn5")
)

print("Project:", project.id)

version = project.version(1)

print("Starting YOLOv8 download...")

dataset = version.download(
    "yolo26",
    location=str(DATASET_DIR)
)

print("\nDOWNLOAD RETURNED")
print("Dataset location:", dataset.location)


Connecting to Roboflow...
loading Roboflow workspace...
loading Roboflow project...
Project: project-60htx/bollard-v2gn5
Starting YOLOv8 download...

DOWNLOAD RETURNED
Dataset location: /content/drive/MyDrive/Hazard_Detection/raw_datasets/obstacle/source_datasets/bollards


In [9]:
import shutil
from pathlib import Path

DATASET_DIR = SOURCE_DIR / "bollards"

# Safety check: never delete a non-empty dataset
if DATASET_DIR.exists():

    if any(DATASET_DIR.iterdir()):
        raise RuntimeError(
            "Destination is not empty. "
            "Stopping to protect existing files."
        )

    DATASET_DIR.rmdir()

print("Destination prepared:", DATASET_DIR)

# Connect to the selected project and version
project = (
    rf.workspace("project-60htx")
    .project("bollard-v2gn5")
)

version = project.version(1)

print("Project:", project.id)
print("Version:", 1)

# Actual download
dataset = version.download(
    "yolo26",
    location=str(DATASET_DIR),
    overwrite=True
)

print("\nDOWNLOAD COMPLETED")
print("Returned location:", dataset.location)

# Immediate verification
download_path = Path(dataset.location)

yaml_files = list(download_path.rglob("*.yaml"))
images = [
    p for p in download_path.rglob("*")
    if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
]
labels = list(download_path.rglob("*.txt"))

print("\n========== VERIFICATION ==========")
print("Directory exists:", download_path.exists())
print("YAML files:", len(yaml_files))
print("Images:", len(images))
print("Label files:", len(labels))

if not yaml_files or not images or not labels:
    raise RuntimeError("Download incomplete. Do not proceed.")

print("\nSUCCESS: Bollards dataset downloaded and verified!")


Destination prepared: /content/drive/MyDrive/Hazard_Detection/raw_datasets/obstacle/source_datasets/bollards
loading Roboflow workspace...
loading Roboflow project...
Project: project-60htx/bollard-v2gn5
Version: 1



Extracting Dataset Version Zip to /content/drive/MyDrive/Hazard_Detection/raw_datasets/obstacle/source_datasets/bollards in yolo26:: 100%|██████████| 2310/2310 [00:28<00:00, 82.19it/s]



DOWNLOAD COMPLETED
Returned location: /content/drive/MyDrive/Hazard_Detection/raw_datasets/obstacle/source_datasets/bollards

========== VERIFICATION ==========
Directory exists: True
YAML files: 1
Images: 1153
Label files: 1155

SUCCESS: Bollards dataset downloaded and verified!


In [10]:

from pathlib import Path
from collections import Counter, defaultdict
import yaml

BOLLARDS_DIR = Path(
    "/content/drive/MyDrive/Hazard_Detection/"
    "raw_datasets/obstacle/source_datasets/bollards"
)

# Find dataset YAML
yaml_files = list(BOLLARDS_DIR.rglob("*.yaml"))

if not yaml_files:
    raise FileNotFoundError("Bollards data.yaml not found")

with open(yaml_files[0], "r") as f:
    config = yaml.safe_load(f)

print("=" * 65)
print("ORIGINAL CLASS MAPPING")
print("=" * 65)

names = config["names"]

if isinstance(names, list):
    class_names = dict(enumerate(names))
else:
    class_names = {int(k): v for k, v in names.items()}

for class_id, name in class_names.items():
    print(f"{class_id}: {name}")

print("\n" + "=" * 65)
print("INSTANCE DISTRIBUTION")
print("=" * 65)

instance_counts = Counter()
image_counts = Counter()
split_images = defaultdict(int)

for split in ["train", "valid", "test"]:

    image_dir = BOLLARDS_DIR / split / "images"
    label_dir = BOLLARDS_DIR / split / "labels"

    if not image_dir.exists():
        print(f"\nMissing image directory: {split}")
        continue

    images = [
        p for p in image_dir.iterdir()
        if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
    ]

    split_images[split] = len(images)

    for image in images:

        label = label_dir / (image.stem + ".txt")

        if not label.exists():
            continue

        image_counts[split] += 1

        with open(label, "r") as f:
            for line in f:

                values = line.strip().split()

                if len(values) != 5:
                    continue

                class_id = int(float(values[0]))
                instance_counts[(split, class_id)] += 1

    print(f"\n{split.upper()}")
    print("Images:", len(images))
    print("Images with labels:", image_counts[split])

    for class_id, name in class_names.items():
        count = instance_counts[(split, class_id)]
        print(f"  {class_id}: {name:<25} {count}")

print("\n" + "=" * 65)
print("TOTAL INSTANCES PER CLASS")
print("=" * 65)

for class_id, name in class_names.items():

    total = sum(
        instance_counts[(split, class_id)]
        for split in ["train", "valid", "test"]
    )

    print(f"{class_id}: {name:<25} {total}")


ORIGINAL CLASS MAPPING
0: bollad1
1: bollard
2: bollard_abnormal
3: bollard_normal

INSTANCE DISTRIBUTION

TRAIN
Images: 1076
Images with labels: 1076
  0: bollad1                   58
  1: bollard                   232
  2: bollard_abnormal          604
  3: bollard_normal            1337

VALID
Images: 77
Images with labels: 77
  0: bollad1                   0
  1: bollard                   0
  2: bollard_abnormal          27
  3: bollard_normal            50

Missing image directory: test

TOTAL INSTANCES PER CLASS
0: bollad1                   58
1: bollard                   232
2: bollard_abnormal          631
3: bollard_normal            1387


In [11]:

# ============================================================
# BOLLARDS: FILTER, SAMPLE, REMAP AND COPY
# ============================================================

from pathlib import Path
from collections import Counter
from PIL import Image
import random
import shutil
import csv
import json

# ------------------------------------------------------------
# 1. PATHS AND SETTINGS
# ------------------------------------------------------------

SOURCE = Path(
    "/content/drive/MyDrive/Hazard_Detection/"
    "raw_datasets/obstacle/source_datasets/bollards"
)

FINAL = Path(
    "/content/drive/MyDrive/Hazard_Detection/"
    "raw_datasets/obstacle/final_dataset"
)

TARGETS = {
    "train": 220,
    "valid": 80,
    "test": 100
}

MASTER_CLASS_ID = 13

ORIGINAL_CLASSES = {
    0: "bollad1",
    1: "bollard",
    2: "bollard_abnormal",
    3: "bollard_normal"
}

SEED = 42

random.seed(SEED)

# ------------------------------------------------------------
# 2. CREATE FINAL DIRECTORIES
# ------------------------------------------------------------

for split in ["train", "valid", "test"]:

    (FINAL / split / "images").mkdir(
        parents=True, exist_ok=True
    )

    (FINAL / split / "labels").mkdir(
        parents=True, exist_ok=True
    )

print("Final dataset directories prepared.")

# ------------------------------------------------------------
# 3. READ SOURCE IMAGES AND ANNOTATIONS
# ------------------------------------------------------------

def read_split(split):

    image_dir = SOURCE / split / "images"
    label_dir = SOURCE / split / "labels"

    records = []

    for image_path in sorted(image_dir.iterdir()):

        if image_path.suffix.lower() not in [
            ".jpg", ".jpeg", ".png"
        ]:
            continue

        label_path = label_dir / (image_path.stem + ".txt")

        if not label_path.exists():
            raise FileNotFoundError(
                f"Missing label for {image_path.name}"
            )

        annotations = []

        with open(label_path, "r") as f:

            for line in f:

                values = line.strip().split()

                if not values:
                    continue

                if len(values) != 5:
                    raise ValueError(
                        f"Invalid annotation: {label_path}"
                    )

                class_id = int(float(values[0]))

                if class_id not in ORIGINAL_CLASSES:
                    raise ValueError(
                        f"Unexpected class ID {class_id} "
                        f"in {label_path}"
                    )

                annotations.append({
                    "original_class_id": class_id,
                    "original_class_name":
                        ORIGINAL_CLASSES[class_id],
                    "coordinates": values[1:]
                })

        # Ignore images containing no accepted bollard boxes
        if annotations:

            records.append({
                "image": image_path,
                "annotations": annotations,
                "counts": Counter(
                    a["original_class_id"]
                    for a in annotations
                )
            })

    return records


train_records = read_split("train")
valid_records = read_split("valid")

print("\nSOURCE INVENTORY")
print("Train images:", len(train_records))
print("Valid images:", len(valid_records))

# ------------------------------------------------------------
# 4. SELECT COMPLETE IMAGES
#    Subset-sum approach: get as close as possible to target
#    WITHOUT exceeding it.
# ------------------------------------------------------------

def select_images(records, target, seed):

    shuffled = records.copy()
    random.Random(seed).shuffle(shuffled)

    # Dynamic programming over total bounding-box count
    previous = {0: None}

    for index, record in enumerate(shuffled):

        count = len(record["annotations"])

        for total in sorted(previous.keys(), reverse=True):

            new_total = total + count

            if new_total <= target and new_total not in previous:
                previous[new_total] = (total, index)

    best_total = max(previous.keys())

    selected_indices = []
    current = best_total

    while current > 0:

        previous_total, index = previous[current]

        selected_indices.append(index)
        current = previous_total

    selected = [
        shuffled[i] for i in selected_indices
    ]

    return selected, best_total


# Select test first, then train from remaining images
test_records, test_count = select_images(
    train_records,
    TARGETS["test"],
    SEED
)

test_image_ids = {
    r["image"].name for r in test_records
}

remaining_train = [
    r for r in train_records
    if r["image"].name not in test_image_ids
]

final_train_records, train_count = select_images(
    remaining_train,
    TARGETS["train"],
    SEED + 1
)

# Keep the complete original validation split
final_valid_records = valid_records

valid_count = sum(
    len(r["annotations"])
    for r in final_valid_records
)

print("\nSELECTION RESULTS")
print("Train:", train_count)
print("Validation:", valid_count)
print("Test:", test_count)

# ------------------------------------------------------------
# 5. CHECK IMAGE-LEVEL SPLIT SEPARATION
# ------------------------------------------------------------

train_ids = {r["image"].name for r in final_train_records}
valid_ids = {r["image"].name for r in final_valid_records}
test_ids = {r["image"].name for r in test_records}

assert not (train_ids & valid_ids)
assert not (train_ids & test_ids)
assert not (valid_ids & test_ids)

print("\nNo image filename overlap between splits.")

# ------------------------------------------------------------
# 6. COPY IMAGES AND REMAP ANNOTATIONS
# ------------------------------------------------------------

manifest_path = FINAL / "collection_manifest.csv"

manifest_columns = [
    "image_filename",
    "source_dataset",
    "source_workspace",
    "source_project",
    "source_version",
    "source_split",
    "final_split",
    "original_class_counts",
    "original_instance_count",
    "final_class_id",
    "final_instance_count"
]

all_jobs = [
    ("train", "train", final_train_records),
    ("valid", "valid", final_valid_records),
    ("test", "train", test_records)
]

manifest_rows = []

for final_split, source_split, records in all_jobs:

    image_output = FINAL / final_split / "images"
    label_output = FINAL / final_split / "labels"

    for record in records:

        source_image = record["image"]

        # Prefix avoids collisions with other datasets later
        output_name = (
            "bollards__" + source_image.name
        )

        output_image = image_output / output_name
        output_label = (
            label_output /
            (Path(output_name).stem + ".txt")
        )

        if output_image.exists() or output_label.exists():
            raise FileExistsError(
                f"Output already exists: {output_name}"
            )

        # Copy image
        shutil.copy2(source_image, output_image)

        # Remap every annotation to master class 13
        with open(output_label, "w") as f:

            for annotation in record["annotations"]:

                coords = annotation["coordinates"]

                f.write(
                    str(MASTER_CLASS_ID) + " " +
                    " ".join(coords) + "\n"
                )

        original_counts = {
            ORIGINAL_CLASSES[class_id]: count
            for class_id, count in record["counts"].items()
        }

        manifest_rows.append({
            "image_filename": output_name,
            "source_dataset": "bollards",
            "source_workspace": "project-60htx",
            "source_project": "bollard-v2gn5",
            "source_version": 1,
            "source_split": source_split,
            "final_split": final_split,
            "original_class_counts": json.dumps(
                original_counts
            ),
            "original_instance_count": len(
                record["annotations"]
            ),
            "final_class_id": MASTER_CLASS_ID,
            "final_instance_count": len(
                record["annotations"]
            )
        })

# Append manifest entries
manifest_exists = manifest_path.exists()

with open(
    manifest_path,
    "a",
    newline=""
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=manifest_columns
    )

    if not manifest_exists:
        writer.writeheader()

    writer.writerows(manifest_rows)

print("\nBOLLARDS COPIED SUCCESSFULLY.")

# ------------------------------------------------------------
# 7. VERIFY FINAL DATASET
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FINAL BOLLARDS VERIFICATION")
print("=" * 65)

for split in ["train", "valid", "test"]:

    label_dir = FINAL / split / "labels"
    image_dir = FINAL / split / "images"

    total_boxes = 0
    class_counts = Counter()

    for label in label_dir.glob("bollards__*.txt"):

        with open(label, "r") as f:

            for line in f:

                values = line.strip().split()

                if not values:
                    continue

                class_id = int(values[0])

                if class_id != MASTER_CLASS_ID:
                    raise ValueError(
                        f"Incorrect class ID in {label}"
                    )

                total_boxes += 1
                class_counts[class_id] += 1

    image_count = len(
        list(image_dir.glob("bollards__*"))
    )

    print(f"\n{split.upper()}")
    print("Images:", image_count)
    print("Bounding boxes:", total_boxes)
    print("Class 13:", class_counts[13])

print("\nManifest:", manifest_path)
print("Bollards integration complete!")


Final dataset directories prepared.


ValueError: Invalid annotation: /content/drive/MyDrive/Hazard_Detection/raw_datasets/obstacle/source_datasets/bollards/train/labels/-28_jpg.rf.4d8966f03ae85b80358fab28e7e9701a.txt

In [12]:

from pathlib import Path

LABEL_PATH = Path(
    "/content/drive/MyDrive/Hazard_Detection/"
    "raw_datasets/obstacle/source_datasets/bollards/"
    "train/labels/-28_jpg.rf.4d8966f03ae85b80358fab28e7e9701a.txt"
)

print("FILE EXISTS:", LABEL_PATH.exists())
print("\nRAW CONTENT:\n")

with open(LABEL_PATH, "r") as f:
    for line_number, line in enumerate(f, 1):
        print(f"{line_number}: {repr(line)}")


FILE EXISTS: True

RAW CONTENT:

1: '2 0.5674307265625 0.378275015625 0.4474398421875 0.88614013125 0.6489340609375 0.8966638093749999 0.7529463890625 0.388742653125 0.5674307265625 0.378275015625\n'
2: '1 0.64765625 0.2328125 0.0828125 0.125'


In [13]:

# ============================================================
# BOLLARDS: FILTER, SAMPLE, CONVERT, REMAP AND COPY
# ============================================================

from pathlib import Path
from collections import Counter
import random
import shutil
import csv
import json

SOURCE = Path(
    "/content/drive/MyDrive/Hazard_Detection/"
    "raw_datasets/obstacle/source_datasets/bollards"
)

FINAL = Path(
    "/content/drive/MyDrive/Hazard_Detection/"
    "raw_datasets/obstacle/final_dataset"
)

TARGETS = {
    "train": 220,
    "valid": 80,
    "test": 100
}

MASTER_CLASS_ID = 13

ORIGINAL_CLASSES = {
    0: "bollad1",
    1: "bollard",
    2: "bollard_abnormal",
    3: "bollard_normal"
}

SEED = 42

# ------------------------------------------------------------
# 1. PREPARE OUTPUT DIRECTORIES
# ------------------------------------------------------------

for split in ["train", "valid", "test"]:

    for folder in ["images", "labels"]:

        output_dir = FINAL / split / folder
        output_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. CONVERT YOLO BOX OR POLYGON TO YOLO BOX
# ------------------------------------------------------------

def convert_annotation(values):

    class_id = int(float(values[0]))
    coords = [float(x) for x in values[1:]]

    if len(coords) == 4:
        # Already YOLO bounding box
        xc, yc, width, height = coords

    elif len(coords) >= 6 and len(coords) % 2 == 0:
        # Polygon: convert to enclosing bounding box

        xs = coords[0::2]
        ys = coords[1::2]

        xmin, xmax = min(xs), max(xs)
        ymin, ymax = min(ys), max(ys)

        xc = (xmin + xmax) / 2
        yc = (ymin + ymax) / 2

        width = xmax - xmin
        height = ymax - ymin

    else:
        raise ValueError(
            f"Unsupported annotation format: {values}"
        )

    # Validate normalized coordinates
    if not (
        0 <= xc <= 1 and
        0 <= yc <= 1 and
        0 < width <= 1 and
        0 < height <= 1
    ):
        raise ValueError(
            f"Invalid normalized bounding box: "
            f"{xc}, {yc}, {width}, {height}"
        )

    return class_id, [xc, yc, width, height]


# ------------------------------------------------------------
# 3. READ SOURCE SPLITS
# ------------------------------------------------------------

def read_split(split):

    image_dir = SOURCE / split / "images"
    label_dir = SOURCE / split / "labels"

    records = []

    for image_path in sorted(image_dir.iterdir()):

        if image_path.suffix.lower() not in [
            ".jpg", ".jpeg", ".png"
        ]:
            continue

        label_path = label_dir / (
            image_path.stem + ".txt"
        )

        if not label_path.exists():
            raise FileNotFoundError(label_path)

        annotations = []

        with open(label_path, "r") as f:

            for line_number, line in enumerate(f, 1):

                values = line.strip().split()

                if not values:
                    continue

                try:
                    class_id, bbox = convert_annotation(values)
                except Exception as e:
                    raise ValueError(
                        f"{label_path}, line {line_number}: {e}"
                    )

                if class_id not in ORIGINAL_CLASSES:
                    raise ValueError(
                        f"Unexpected class ID {class_id}"
                    )

                annotations.append({
                    "original_class_id": class_id,
                    "original_class_name":
                        ORIGINAL_CLASSES[class_id],
                    "bbox": bbox
                })

        if annotations:

            records.append({
                "image": image_path,
                "annotations": annotations
            })

    return records


train_records = read_split("train")
valid_records = read_split("valid")

print("TRAIN IMAGES:", len(train_records))
print("VALID IMAGES:", len(valid_records))


# ------------------------------------------------------------
# 4. SELECT COMPLETE IMAGES WITHOUT EXCEEDING TARGET
# ------------------------------------------------------------

def select_images(records, target, seed):

    shuffled = records.copy()
    random.Random(seed).shuffle(shuffled)

    # Dynamic programming
    previous = {0: None}

    for index, record in enumerate(shuffled):

        count = len(record["annotations"])

        for total in sorted(
            list(previous.keys()), reverse=True
        ):

            new_total = total + count

            if new_total <= target and new_total not in previous:
                previous[new_total] = (total, index)

    best_total = max(previous.keys())

    selected_indices = []
    current = best_total

    while current > 0:

        previous_total, index = previous[current]

        selected_indices.append(index)
        current = previous_total

    selected = [
        shuffled[i] for i in selected_indices
    ]

    return selected, best_total


# Test first, then train from remaining images
test_records, test_count = select_images(
    train_records, TARGETS["test"], SEED
)

test_names = {
    r["image"].name for r in test_records
}

remaining_train = [
    r for r in train_records
    if r["image"].name not in test_names
]

final_train_records, train_count = select_images(
    remaining_train, TARGETS["train"], SEED + 1
)

final_valid_records = valid_records

valid_count = sum(
    len(r["annotations"]) for r in final_valid_records
)

print("\nSELECTED BOX COUNTS")
print("Train:", train_count)
print("Validation:", valid_count)
print("Test:", test_count)


# ------------------------------------------------------------
# 5. CHECK SPLIT SEPARATION
# ------------------------------------------------------------

train_names = {r["image"].name for r in final_train_records}
valid_names = {r["image"].name for r in final_valid_records}

assert not (train_names & valid_names)
assert not (train_names & test_names)
assert not (valid_names & test_names)

print("\nIMAGE SPLIT SEPARATION VERIFIED.")


# ------------------------------------------------------------
# 6. COPY AND REMAP
# ------------------------------------------------------------

manifest_path = FINAL / "collection_manifest.csv"

manifest_columns = [
    "image_filename",
    "source_dataset",
    "source_workspace",
    "source_project",
    "source_version",
    "source_split",
    "final_split",
    "original_class_counts",
    "original_instance_count",
    "final_class_id",
    "final_instance_count",
    "polygon_conversions"
]

jobs = [
    ("train", "train", final_train_records),
    ("valid", "valid", final_valid_records),
    ("test", "train", test_records)
]

manifest_rows = []

for final_split, source_split, records in jobs:

    for record in records:

        source_image = record["image"]

        output_name = "bollards__" + source_image.name

        output_image = FINAL / final_split / "images" / output_name

        output_label = (
            FINAL / final_split / "labels" /
            (Path(output_name).stem + ".txt")
        )

        if output_image.exists() or output_label.exists():
            raise FileExistsError(output_name)

        shutil.copy2(source_image, output_image)

        original_counts = Counter()
        polygon_conversions = 0

        with open(output_label, "w") as f:

            for annotation in record["annotations"]:

                original_id = annotation["original_class_id"]
                original_counts[
                    ORIGINAL_CLASSES[original_id]
                ] += 1

                bbox = annotation["bbox"]

                f.write(
                    f"{MASTER_CLASS_ID} " +
                    " ".join(f"{v:.10f}" for v in bbox) +
                    "\n"
                )

        # Count polygon conversions from original label file
        source_label = (
            SOURCE / source_split / "labels" /
            (source_image.stem + ".txt")
        )

        with open(source_label, "r") as f:
            for line in f:
                values = line.strip().split()
                if values and len(values) > 5:
                    polygon_conversions += 1

        manifest_rows.append({
            "image_filename": output_name,
            "source_dataset": "bollards",
            "source_workspace": "project-60htx",
            "source_project": "bollard-v2gn5",
            "source_version": 1,
            "source_split": source_split,
            "final_split": final_split,
            "original_class_counts": json.dumps(
                dict(original_counts)
            ),
            "original_instance_count": len(
                record["annotations"]
            ),
            "final_class_id": MASTER_CLASS_ID,
            "final_instance_count": len(
                record["annotations"]
            ),
            "polygon_conversions": polygon_conversions
        })


# ------------------------------------------------------------
# 7. WRITE MANIFEST
# ------------------------------------------------------------

if manifest_path.exists():
    existing_rows = []

    with open(manifest_path, "r") as f:
        existing_rows = list(csv.DictReader(f))

    manifest_rows = existing_rows + manifest_rows

with open(manifest_path, "w", newline="") as f:

    writer = csv.DictWriter(
        f, fieldnames=manifest_columns
    )

    writer.writeheader()
    writer.writerows(manifest_rows)


# ------------------------------------------------------------
# 8. FINAL VERIFICATION
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("BOLLARDS FINAL DATASET")
print("=" * 65)

for split in ["train", "valid", "test"]:

    label_dir = FINAL / split / "labels"

    total = 0

    for label in label_dir.glob("bollards__*.txt"):

        with open(label, "r") as f:

            for line in f:

                values = line.strip().split()

                if not values:
                    continue

                assert int(values[0]) == MASTER_CLASS_ID
                assert len(values) == 5

                total += 1

    print(f"{split.upper()}: {total} instances")

print("\nManifest:", manifest_path)
print("BOLLARDS PROCESSING COMPLETED.")


TRAIN IMAGES: 1072
VALID IMAGES: 77

SELECTED BOX COUNTS
Train: 220
Validation: 77
Test: 100

IMAGE SPLIT SEPARATION VERIFIED.

BOLLARDS FINAL DATASET
TRAIN: 220 instances
VALID: 77 instances
TEST: 100 instances

Manifest: /content/drive/MyDrive/Hazard_Detection/raw_datasets/obstacle/final_dataset/collection_manifest.csv
BOLLARDS PROCESSING COMPLETED.


Collecting the barricades dataset from here

In [14]:

from pathlib import Path

BARRICADES_DIR = SOURCE_DIR / "barricades"

# Protect existing data
if BARRICADES_DIR.exists():

    if any(BARRICADES_DIR.iterdir()):
        raise FileExistsError(
            f"Destination already contains files: {BARRICADES_DIR}"
        )

    BARRICADES_DIR.rmdir()

print("Connecting to Roboflow...")

barricade_project = (
    rf.workspace("chandana-k-v-d78vp")
    .project("barricades-qdaov")
)

print("Project:", barricade_project.id)

barricade_version = barricade_project.version(3)

print("Downloading version 3...")

barricade_dataset = barricade_version.download(
    "yolov8",
    location=str(BARRICADES_DIR),
    overwrite=True
)

print("\nDOWNLOAD COMPLETED")
print("Returned location:", barricade_dataset.location)


Connecting to Roboflow...
loading Roboflow workspace...
loading Roboflow project...
Project: chandana-k-v-d78vp/barricades-qdaov



Extracting Dataset Version Zip to /content/drive/MyDrive/Hazard_Detection/raw_datasets/obstacle/source_datasets/barricades in yolov8:: 100%|██████████| 2162/2162 [00:29<00:00, 73.14it/s]


DOWNLOAD COMPLETED
Returned location: /content/drive/MyDrive/Hazard_Detection/raw_datasets/obstacle/source_datasets/barricades


In [15]:

from pathlib import Path
import yaml

download_path = Path(barricade_dataset.location)

yaml_files = (
    list(download_path.rglob("*.yaml")) +
    list(download_path.rglob("*.yml"))
)

images = [
    p for p in download_path.rglob("*")
    if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
]

labels = list(download_path.rglob("*.txt"))

print("=" * 60)
print("BARRICADES DOWNLOAD VERIFICATION")
print("=" * 60)

print("Location:", download_path)
print("Directory exists:", download_path.exists())
print("YAML files:", len(yaml_files))
print("Images:", len(images))
print("Labels:", len(labels))

if not yaml_files or not images or not labels:
    raise RuntimeError("Download incomplete. Stop here.")

for yaml_file in yaml_files:
    print("\nYAML:", yaml_file)
    print(yaml_file.read_text())

print("\nDOWNLOAD VERIFIED SUCCESSFULLY!")


BARRICADES DOWNLOAD VERIFICATION
Location: /content/drive/MyDrive/Hazard_Detection/raw_datasets/obstacle/source_datasets/barricades
Directory exists: True
YAML files: 1
Images: 1075
Labels: 1077

YAML: /content/drive/MyDrive/Hazard_Detection/raw_datasets/obstacle/source_datasets/barricades/data.yaml
names:
- barricade
nc: 1
roboflow:
  license: CC BY 4.0
  project: barricades-qdaov
  url: https://universe.roboflow.com/chandana-k-v-d78vp/barricades-qdaov/dataset/3
  version: 3
  workspace: chandana-k-v-d78vp
test: ../test/images
train: ../train/images
val: ../valid/images


DOWNLOAD VERIFIED SUCCESSFULLY!


In [16]:

from collections import Counter

config = yaml.safe_load(yaml_files[0].read_text())

print("=" * 60)
print("ORIGINAL CLASS MAPPING")
print("=" * 60)

names = config["names"]

if isinstance(names, list):
    class_names = dict(enumerate(names))
else:
    class_names = {
        int(k): v for k, v in names.items()
    }

for class_id, name in class_names.items():
    print(f"{class_id}: {name}")

print("\n" + "=" * 60)
print("INSTANCE DISTRIBUTION")
print("=" * 60)

for split in ["train", "valid", "test"]:

    image_dir = download_path / split / "images"
    label_dir = download_path / split / "labels"

    if not image_dir.exists():
        print(f"\n{split.upper()}: Missing")
        continue

    split_counts = Counter()
    image_count = 0
    labeled_images = 0

    for image in image_dir.iterdir():

        if image.suffix.lower() not in [
            ".jpg", ".jpeg", ".png"
        ]:
            continue

        image_count += 1

        label = label_dir / (image.stem + ".txt")

        if not label.exists():
            continue

        labeled_images += 1

        with open(label, "r") as f:

            for line in f:

                values = line.strip().split()

                if values:
                    split_counts[int(float(values[0]))] += 1

    print(f"\n{split.upper()}")
    print("Images:", image_count)
    print("Images with labels:", labeled_images)

    for class_id, name in class_names.items():
        print(
            f"{class_id}: {name:<25} "
            f"{split_counts[class_id]}"
        )


ORIGINAL CLASS MAPPING
0: barricade

INSTANCE DISTRIBUTION

TRAIN
Images: 805
Images with labels: 805
0: barricade                 1035

VALID
Images: 135
Images with labels: 135
0: barricade                 228

TEST
Images: 135
Images with labels: 135
0: barricade                 208


In [17]:

from collections import Counter
from pathlib import Path

print("=" * 65)
print("BARRICADE ANNOTATION FORMAT INSPECTION")
print("=" * 65)

for split in ["train", "valid", "test"]:

    label_dir = download_path / split / "labels"

    line_lengths = Counter()
    instance_counts = Counter()
    empty_labels = 0
    malformed_lines = 0
    samples = []

    for label_file in sorted(label_dir.glob("*.txt")):

        lines = [
            line.strip()
            for line in label_file.read_text().splitlines()
            if line.strip()
        ]

        if not lines:
            empty_labels += 1
            continue

        instance_counts[len(lines)] += 1

        for line in lines:
            values = line.split()
            line_lengths[len(values)] += 1

            if len(values) < 5:
                malformed_lines += 1

            if len(samples) < 5:
                samples.append((label_file.name, line))

    print(f"\n{split.upper()}")
    print("Annotation line lengths:", dict(line_lengths))
    print("Images per instance count:", dict(instance_counts))
    print("Empty label files:", empty_labels)
    print("Malformed lines:", malformed_lines)

    print("\nSample annotations:")
    for filename, line in samples:
        print(filename, ":", line)

print("\nInspection complete.")


BARRICADE ANNOTATION FORMAT INSPECTION

TRAIN
Annotation line lengths: {13: 71, 15: 42, 59: 5, 19: 22, 39: 11, 25: 18, 27: 9, 75: 4, 33: 6, 53: 6, 45: 12, 5: 539, 49: 8, 43: 4, 21: 22, 61: 7, 17: 28, 11: 81, 107: 1, 63: 7, 41: 10, 23: 9, 121: 2, 67: 7, 77: 4, 47: 10, 55: 7, 31: 12, 141: 1, 79: 3, 71: 2, 81: 3, 99: 2, 65: 2, 85: 3, 29: 11, 89: 4, 9: 1, 37: 7, 87: 3, 51: 5, 83: 4, 35: 5, 117: 1, 69: 4, 73: 3, 101: 2, 91: 4, 57: 1}
Images per instance count: {1: 677, 2: 61, 4: 15, 9: 1, 7: 2, 3: 19, 5: 7, 6: 4, 8: 1, 14: 1, 15: 1}
Empty label files: 16
Malformed lines: 0

Sample annotations:
001_RBI-image-JOC039260I01_jpg.rf.a445a5337ab77862b025d4f05847392e.txt : 0 0.9808319734375001 0.58987119375 0.9798123984375 0.569379390625 0.0061174546875 0.3586065578125 0.0010195765625 0.7933255265625 0.3140293640625 0.9909250578125001 0.9808319734375001 0.58987119375
0829d286c5f1abbae4961277521a24d2_jpg.rf.0a69ffb0c399c94a525135685edfb366.txt : 0 0.1 1 0.2109375 0.923444975 0.209375 0.6387559796875

In [18]:

from collections import Counter
from pathlib import Path
import math

print("=" * 70)
print("BARRICADE ANNOTATION VALIDATION")
print("=" * 70)

overall = Counter()

for split in ["train", "valid", "test"]:

    label_dir = download_path / split / "labels"
    image_dir = download_path / split / "images"

    stats = Counter()
    invalid_examples = []
    image_stems = {
        p.stem for p in image_dir.iterdir()
        if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
    }

    label_stems = {
        p.stem for p in label_dir.glob("*.txt")
    }

    missing_labels = image_stems - label_stems
    orphan_labels = label_stems - image_stems

    for label_file in sorted(label_dir.glob("*.txt")):

        lines = [
            line.strip()
            for line in label_file.read_text().splitlines()
            if line.strip()
        ]

        if not lines:
            stats["empty_label_files"] += 1
            continue

        for line in lines:

            values = line.split()

            try:
                numbers = [float(x) for x in values]
                class_id = int(numbers[0])
                coords = numbers[1:]

                if not all(math.isfinite(x) for x in numbers):
                    raise ValueError("Non-finite value")

                if class_id != 0:
                    raise ValueError("Unexpected class ID")

                if len(coords) == 4:
                    stats["bounding_boxes"] += 1

                    xc, yc, w, h = coords

                    if not (
                        0 <= xc <= 1 and
                        0 <= yc <= 1 and
                        0 < w <= 1 and
                        0 < h <= 1
                    ):
                        raise ValueError("Invalid bbox coordinates")

                elif len(coords) >= 6 and len(coords) % 2 == 0:
                    stats["polygons"] += 1

                    xs = coords[0::2]
                    ys = coords[1::2]

                    if not all(0 <= x <= 1 for x in xs):
                        raise ValueError("Invalid polygon X coordinate")

                    if not all(0 <= y <= 1 for y in ys):
                        raise ValueError("Invalid polygon Y coordinate")

                    if max(xs) <= min(xs) or max(ys) <= min(ys):
                        raise ValueError("Degenerate polygon")

                else:
                    raise ValueError(
                        f"Unexpected coordinate count: {len(coords)}"
                    )

            except Exception as e:
                stats["invalid_annotations"] += 1

                if len(invalid_examples) < 5:
                    invalid_examples.append(
                        (label_file.name, line[:150], str(e))
                    )

    stats["images_without_label_files"] = len(missing_labels)
    stats["orphan_label_files"] = len(orphan_labels)

    overall.update(stats)

    print(f"\n{split.upper()}")
    print("-" * 45)

    for key, value in stats.items():
        print(f"{key:<32}: {value}")

    if invalid_examples:
        print("\nINVALID EXAMPLES:")
        for example in invalid_examples:
            print(example)

print("\n" + "=" * 70)
print("OVERALL TOTALS")
print("=" * 70)

for key, value in overall.items():
    print(f"{key:<32}: {value}")

print("\nValidation inspection completed.")


BARRICADE ANNOTATION VALIDATION

TRAIN
---------------------------------------------
polygons                        : 496
bounding_boxes                  : 539
empty_label_files               : 16
images_without_label_files      : 0
orphan_label_files              : 0

VALID
---------------------------------------------
polygons                        : 68
bounding_boxes                  : 160
empty_label_files               : 3
images_without_label_files      : 0
orphan_label_files              : 0

TEST
---------------------------------------------
polygons                        : 72
bounding_boxes                  : 136
empty_label_files               : 2
images_without_label_files      : 0
orphan_label_files              : 0

OVERALL TOTALS
polygons                        : 636
bounding_boxes                  : 835
empty_label_files               : 21
images_without_label_files      : 0
orphan_label_files              : 0

Validation inspection completed.


In [19]:

import random
from pathlib import Path
from collections import Counter

TARGETS = {
    "train": 220,
    "valid": 80,
    "test": 100
}

RANDOM_SEED = 42
rng = random.Random(RANDOM_SEED)

# Protect existing final dataset
for split in ["train", "valid", "test"]:
    for subfolder in ["images", "labels"]:
        path = FINAL / split / subfolder

        if not path.exists():
            raise FileNotFoundError(
                f"Expected final dataset folder missing: {path}"
            )

# Protect against accidentally processing barricades twice
manifest_path = FINAL / "collection_manifest.csv"

if manifest_path.exists():
    import pandas as pd

    existing_manifest = pd.read_csv(manifest_path)

    if (
        "source_dataset" in existing_manifest.columns
        and existing_manifest["source_dataset"]
        .astype(str).str.contains("barricades", case=False).any()
    ):
        raise RuntimeError(
            "Barricades already appear in the manifest. "
            "Stop to avoid duplicate integration."
        )

    if (
        "image_filename" in existing_manifest.columns
        and existing_manifest["image_filename"]
        .astype(str).str.startswith("barricades__").any()
    ):
        raise RuntimeError(
            "Barricade-prefixed filenames already exist in manifest."
        )

# Check final folders for existing barricade files
for split in ["train", "valid", "test"]:
    for subfolder in ["images", "labels"]:
        existing = list(
            (FINAL / split / subfolder).glob("barricades__*")
        )

        if existing:
            raise RuntimeError(
                f"Existing barricade files found in {split}/{subfolder}"
            )

# Select whole images
BARRICADE_SELECTION = {}

print("=" * 65)
print("BARRICADE IMAGE SELECTION PLAN")
print("=" * 65)

for split, target in TARGETS.items():

    image_dir = download_path / split / "images"
    label_dir = download_path / split / "labels"

    candidates = []

    for image_path in image_dir.iterdir():

        if image_path.suffix.lower() not in [
            ".jpg", ".jpeg", ".png"
        ]:
            continue

        label_path = label_dir / f"{image_path.stem}.txt"

        if not label_path.exists():
            continue

        lines = [
            line.strip()
            for line in label_path.read_text().splitlines()
            if line.strip()
        ]

        if not lines:
            continue

        candidates.append({
            "image": image_path,
            "label": label_path,
            "instances": len(lines)
        })

    rng.shuffle(candidates)

    selected = []
    total_instances = 0

    for item in candidates:

        if total_instances >= target:
            break

        selected.append(item)
        total_instances += item["instances"]

    if total_instances < target:
        raise RuntimeError(
            f"Insufficient instances in {split}: "
            f"{total_instances}/{target}"
        )

    BARRICADE_SELECTION[split] = selected

    print(f"\n{split.upper()}")
    print("Target instances:", target)
    print("Selected images:", len(selected))
    print("Selected instances:", total_instances)
    print("Overshoot:", total_instances - target)

print("\n" + "=" * 65)
print("SELECTION PLAN READY — NO FILES COPIED")
print("=" * 65)


BARRICADE IMAGE SELECTION PLAN

TRAIN
Target instances: 220
Selected images: 163
Selected instances: 220
Overshoot: 0

VALID
Target instances: 80
Selected images: 48
Selected instances: 87
Overshoot: 7

TEST
Target instances: 100
Selected images: 78
Selected instances: 100
Overshoot: 0

SELECTION PLAN READY — NO FILES COPIED


In [20]:

import random
from pathlib import Path

TARGETS = {
    "train": 220,
    "valid": 80,
    "test": 100
}

rng = random.Random(42)

BARRICADE_SELECTION = {}

print("=" * 65)
print("BARRICADE BBOX-ONLY IMAGE SELECTION")
print("=" * 65)

for split, target in TARGETS.items():

    image_dir = download_path / split / "images"
    label_dir = download_path / split / "labels"

    candidates = []

    for image_path in image_dir.iterdir():

        if image_path.suffix.lower() not in [
            ".jpg", ".jpeg", ".png"
        ]:
            continue

        label_path = label_dir / f"{image_path.stem}.txt"

        if not label_path.exists():
            continue

        bbox_lines = []
        polygon_count = 0

        for line in label_path.read_text().splitlines():

            values = line.strip().split()

            if not values:
                continue

            if len(values) == 5:
                bbox_lines.append(line.strip())

            elif len(values) > 5:
                polygon_count += 1

        # Only images containing at least one original bbox
        if not bbox_lines:
            continue

        candidates.append({
            "image": image_path,
            "label": label_path,
            "bbox_count": len(bbox_lines),
            "polygon_count": polygon_count
        })

    rng.shuffle(candidates)

    selected = []
    total_instances = 0
    ignored_polygons = 0

    for item in candidates:

        if total_instances >= target:
            break

        selected.append(item)

        total_instances += item["bbox_count"]
        ignored_polygons += item["polygon_count"]

    if total_instances < target:
        raise RuntimeError(
            f"Insufficient bbox instances in {split}: "
            f"{total_instances}/{target}"
        )

    BARRICADE_SELECTION[split] = selected

    print(f"\n{split.upper()}")
    print("-" * 45)
    print("Eligible images:", len(candidates))
    print("Selected images:", len(selected))
    print("Target bbox instances:", target)
    print("Selected bbox instances:", total_instances)
    print("Overshoot:", total_instances - target)
    print("Polygons ignored in selected images:", ignored_polygons)

print("\n" + "=" * 65)
print("BBOX-ONLY SELECTION PLAN READY")
print("No files copied or modified.")
print("=" * 65)


BARRICADE BBOX-ONLY IMAGE SELECTION

TRAIN
---------------------------------------------
Eligible images: 411
Selected images: 175
Target bbox instances: 220
Selected bbox instances: 222
Overshoot: 2
Polygons ignored in selected images: 1

VALID
---------------------------------------------
Eligible images: 85
Selected images: 54
Target bbox instances: 80
Selected bbox instances: 80
Overshoot: 0
Polygons ignored in selected images: 0

TEST
---------------------------------------------
Eligible images: 78
Selected images: 57
Target bbox instances: 100
Selected bbox instances: 103
Overshoot: 3
Polygons ignored in selected images: 0

BBOX-ONLY SELECTION PLAN READY
No files copied or modified.


In [21]:

import shutil
import json
import pandas as pd
from pathlib import Path

# ==========================================
# CONFIGURATION
# ==========================================

MASTER_CLASS_ID = 13

SOURCE_WORKSPACE = "chandana-k-v-d78vp"
SOURCE_PROJECT = "barricades-qdaov"
SOURCE_VERSION = 3
SOURCE_NAME = "barricades"

manifest_path = FINAL / "collection_manifest.csv"

EXPECTED_COLUMNS = [
    "image_filename",
    "source_dataset",
    "source_workspace",
    "source_project",
    "source_version",
    "source_split",
    "final_split",
    "original_class_counts",
    "original_instance_count",
    "final_class_id",
    "final_instance_count",
    "polygon_conversions"
]

# ==========================================
# SAFETY CHECKS
# ==========================================

if not manifest_path.exists():
    raise FileNotFoundError(
        "Existing bollard manifest not found. Stopping."
    )

existing_manifest = pd.read_csv(manifest_path)

if list(existing_manifest.columns) != EXPECTED_COLUMNS:
    raise ValueError(
        "Manifest columns do not match expected schema.\n"
        f"Found: {list(existing_manifest.columns)}"
    )

if existing_manifest["source_dataset"].astype(str).str.lower().eq(
    SOURCE_NAME
).any():
    raise RuntimeError(
        "Barricades already exist in the manifest. Stopping."
    )

for split in ["train", "valid", "test"]:
    for folder in ["images", "labels"]:

        output_dir = FINAL / split / folder

        if not output_dir.exists():
            raise FileNotFoundError(output_dir)

        if list(output_dir.glob("barricades__*")):
            raise RuntimeError(
                f"Barricade-prefixed files already exist in {output_dir}"
            )

# ==========================================
# PREPARE OUTPUT AND MANIFEST RECORDS
# ==========================================

manifest_rows = []
copy_operations = []

print("=" * 65)
print("PREPARING BARRICADE INTEGRATION")
print("=" * 65)

for split, selected_images in BARRICADE_SELECTION.items():

    for item in selected_images:

        source_image = item["image"]
        source_label = item["label"]

        output_filename = (
            f"barricades__{source_image.name}"
        )

        output_labelname = (
            f"barricades__{source_image.stem}.txt"
        )

        destination_image = (
            FINAL / split / "images" / output_filename
        )

        destination_label = (
            FINAL / split / "labels" / output_labelname
        )

        # Read original annotations
        original_lines = [
            line.strip()
            for line in source_label.read_text().splitlines()
            if line.strip()
        ]

        bbox_lines = []
        polygon_count = 0

        for line in original_lines:

            values = line.split()

            if len(values) == 5:

                # Original YOLO bbox:
                # class xc yc width height

                if int(float(values[0])) != 0:
                    raise ValueError(
                        f"Unexpected class in {source_label}"
                    )

                # Remap source class 0 -> master class 13
                remapped_line = " ".join(
                    [str(MASTER_CLASS_ID)] + values[1:]
                )

                bbox_lines.append(remapped_line)

            elif len(values) > 5:
                polygon_count += 1

            else:
                raise ValueError(
                    f"Malformed annotation in {source_label}"
                )

        if len(bbox_lines) != item["bbox_count"]:
            raise RuntimeError(
                f"Selection count mismatch: {source_label}"
            )

        if not bbox_lines:
            raise RuntimeError(
                f"No bounding boxes found: {source_label}"
            )

        # Ensure source and selection agree about polygon count
        if polygon_count != item["polygon_count"]:
            raise RuntimeError(
                f"Polygon count mismatch: {source_label}"
            )

        # Keep complete-image source metadata
        original_instance_count = (
            len(bbox_lines) + polygon_count
        )

        manifest_rows.append({
            "image_filename": output_filename,
            "source_dataset": SOURCE_NAME,
            "source_workspace": SOURCE_WORKSPACE,
            "source_project": SOURCE_PROJECT,
            "source_version": SOURCE_VERSION,
            "source_split": split,
            "final_split": split,
            "original_class_counts": json.dumps({
                "barricade": original_instance_count
            }),
            "original_instance_count": original_instance_count,
            "final_class_id": MASTER_CLASS_ID,
            "final_instance_count": len(bbox_lines),
            "polygon_conversions": 0
        })

        copy_operations.append({
            "source_image": source_image,
            "destination_image": destination_image,
            "destination_label": destination_label,
            "label_content": "\n".join(bbox_lines) + "\n"
        })

# ==========================================
# CHECK ALL DESTINATIONS BEFORE WRITING
# ==========================================

for operation in copy_operations:

    if operation["destination_image"].exists():
        raise FileExistsError(
            operation["destination_image"]
        )

    if operation["destination_label"].exists():
        raise FileExistsError(
            operation["destination_label"]
        )

print("Images prepared:", len(copy_operations))
print("Manifest rows prepared:", len(manifest_rows))

# ==========================================
# COPY IMAGES AND WRITE REMAPPED LABELS
# ==========================================

for operation in copy_operations:

    shutil.copy2(
        operation["source_image"],
        operation["destination_image"]
    )

    operation["destination_label"].write_text(
        operation["label_content"]
    )

# ==========================================
# APPEND TO EXISTING MANIFEST
# ==========================================

new_manifest = pd.DataFrame(
    manifest_rows,
    columns=EXPECTED_COLUMNS
)

updated_manifest = pd.concat(
    [existing_manifest, new_manifest],
    ignore_index=True
)

# Save to temporary file first, then replace manifest
temporary_manifest = (
    manifest_path.parent / "collection_manifest_temp.csv"
)

updated_manifest.to_csv(
    temporary_manifest,
    index=False
)

temporary_manifest.replace(manifest_path)

print("\n" + "=" * 65)
print("BARRICADE INTEGRATION COMPLETED")
print("=" * 65)

for split in ["train", "valid", "test"]:

    split_rows = new_manifest[
        new_manifest["final_split"] == split
    ]

    print(f"\n{split.upper()}")
    print("Images added:", len(split_rows))
    print(
        "Instances added:",
        split_rows["final_instance_count"].sum()
    )

print("\nTotal images added:", len(new_manifest))
print(
    "Total bbox instances added:",
    new_manifest["final_instance_count"].sum()
)

print("\nUpdated manifest:", manifest_path)


PREPARING BARRICADE INTEGRATION
Images prepared: 286
Manifest rows prepared: 286

BARRICADE INTEGRATION COMPLETED

TRAIN
Images added: 175
Instances added: 222

VALID
Images added: 54
Instances added: 80

TEST
Images added: 57
Instances added: 103

Total images added: 286
Total bbox instances added: 405

Updated manifest: /content/drive/MyDrive/Hazard_Detection/raw_datasets/obstacle/final_dataset/collection_manifest.csv


In [22]:

from collections import Counter
import pandas as pd

print("=" * 65)
print("BARRICADE FINAL INTEGRATION VERIFICATION")
print("=" * 65)

manifest = pd.read_csv(manifest_path)

barricade_rows = manifest[
    manifest["source_dataset"].astype(str).str.lower()
    == "barricades"
]

for split in ["train", "valid", "test"]:

    image_dir = FINAL / split / "images"
    label_dir = FINAL / split / "labels"

    rows = barricade_rows[
        barricade_rows["final_split"] == split
    ]

    total_instances = 0
    class_counts = Counter()
    missing_pairs = 0
    invalid_labels = 0

    for _, row in rows.iterrows():

        image_path = image_dir / row["image_filename"]

        label_path = (
            label_dir /
            (Path(row["image_filename"]).stem + ".txt")
        )

        if not image_path.exists() or not label_path.exists():
            missing_pairs += 1
            continue

        for line in label_path.read_text().splitlines():

            values = line.split()

            if len(values) != 5:
                invalid_labels += 1
                continue

            class_id = int(values[0])
            coords = [float(x) for x in values[1:]]

            if class_id != MASTER_CLASS_ID:
                invalid_labels += 1
                continue

            xc, yc, w, h = coords

            if not (
                0 <= xc <= 1 and
                0 <= yc <= 1 and
                0 < w <= 1 and
                0 < h <= 1
            ):
                invalid_labels += 1
                continue

            class_counts[class_id] += 1
            total_instances += 1

    print(f"\n{split.upper()}")
    print("Manifest images:", len(rows))
    print("Verified instances:", total_instances)
    print("Class distribution:", dict(class_counts))
    print("Missing image-label pairs:", missing_pairs)
    print("Invalid labels:", invalid_labels)

    assert missing_pairs == 0
    assert invalid_labels == 0
    assert total_instances == rows["final_instance_count"].sum()

print("\n" + "=" * 65)
print("BARRICADE VERIFICATION COMPLETED")
print("Expected final class: 13 (obstacle)")
print("=" * 65)

print("\nManifest source counts:")
print(manifest["source_dataset"].value_counts())


BARRICADE FINAL INTEGRATION VERIFICATION

TRAIN
Manifest images: 175
Verified instances: 222
Class distribution: {13: 222}
Missing image-label pairs: 0
Invalid labels: 0

VALID
Manifest images: 54
Verified instances: 80
Class distribution: {13: 80}
Missing image-label pairs: 0
Invalid labels: 0

TEST
Manifest images: 57
Verified instances: 103
Class distribution: {13: 103}
Missing image-label pairs: 0
Invalid labels: 0

BARRICADE VERIFICATION COMPLETED
Expected final class: 13 (obstacle)

Manifest source counts:
source_dataset
barricades    286
bollards      226
Name: count, dtype: int64


Collecting the trashcan dataset from here

In [23]:

TRASH_MAIN_DIR = SOURCE_DIR / "trashcan_main"

if TRASH_MAIN_DIR.exists():

    if any(TRASH_MAIN_DIR.iterdir()):
        raise FileExistsError(
            f"Destination already contains files: {TRASH_MAIN_DIR}"
        )

    TRASH_MAIN_DIR.rmdir()

trash_main_project = (
    rf.workspace("group10textdetect")
    .project("trashcan-detect")
)

print("PROJECT:", trash_main_project.id)

trash_main_version = trash_main_project.version(6)

trash_main_dataset = trash_main_version.download(
    "yolov8",
    location=str(TRASH_MAIN_DIR),
    overwrite=True
)

print("\nDOWNLOAD COMPLETED")
print("Returned location:", trash_main_dataset.location)


loading Roboflow workspace...
loading Roboflow project...
PROJECT: group10textdetect/trashcan-detect
Exporting format yolov8 in progress : 0.1%
Version export complete for yolov8 format



Extracting Dataset Version Zip to /content/drive/MyDrive/Hazard_Detection/raw_datasets/obstacle/source_datasets/trashcan_main in yolov8:: 100%|██████████| 245/245 [00:02<00:00, 92.73it/s]


DOWNLOAD COMPLETED
Returned location: /content/drive/MyDrive/Hazard_Detection/raw_datasets/obstacle/source_datasets/trashcan_main


In [24]:

from pathlib import Path

trash_main_path = Path(trash_main_dataset.location)

trash_main_yaml = list(trash_main_path.rglob("*.yaml"))

trash_main_images = [
    p for p in trash_main_path.rglob("*")
    if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
]

trash_main_labels = list(trash_main_path.rglob("*.txt"))

print("=" * 60)
print("PRIMARY TRASHCAN DATASET VERIFICATION")
print("=" * 60)

print("Location:", trash_main_path)
print("YAML files:", len(trash_main_yaml))
print("Images:", len(trash_main_images))
print("Labels:", len(trash_main_labels))

if not trash_main_yaml or not trash_main_images or not trash_main_labels:
    raise RuntimeError("Primary download incomplete!")

for yaml_file in trash_main_yaml:
    print("\nYAML:", yaml_file)
    print(yaml_file.read_text())

print("\nPRIMARY DATASET VERIFIED")


PRIMARY TRASHCAN DATASET VERIFICATION
Location: /content/drive/MyDrive/Hazard_Detection/raw_datasets/obstacle/source_datasets/trashcan_main
YAML files: 1
Images: 120
Labels: 122

YAML: /content/drive/MyDrive/Hazard_Detection/raw_datasets/obstacle/source_datasets/trashcan_main/data.yaml
names:
- bench
- bicycle
- car
- cat
- chair
- dog
- door
- electric fan
- gate
- monoblock
- motor
- pedestrian
- person
- plastic
- pole
- potted
- stair
- swivel
- table
- text
- trash can
nc: 21
roboflow:
  license: CC BY 4.0
  project: trashcan-detect
  url: https://universe.roboflow.com/group10textdetect/trashcan-detect/dataset/6
  version: 6
  workspace: group10textdetect
test: ../test/images
train: ../train/images
val: ../valid/images


PRIMARY DATASET VERIFIED


In [25]:

import yaml
from collections import Counter
from pathlib import Path

config = yaml.safe_load(trash_main_yaml[0].read_text())

names = config["names"]

class_names = (
    dict(enumerate(names))
    if isinstance(names, list)
    else {int(k): v for k, v in names.items()}
)

TRASH_CLASS_ID = next(
    (
        class_id
        for class_id, name in class_names.items()
        if name.strip().lower() == "trash can"
    ),
    None
)

if TRASH_CLASS_ID is None:
    raise ValueError("Could not find trash can class!")

print("TRASH CAN CLASS ID:", TRASH_CLASS_ID)

print("\n" + "=" * 65)
print("PRIMARY TRASHCAN INSTANCE DISTRIBUTION")
print("=" * 65)

for split in ["train", "valid", "test"]:

    image_dir = trash_main_path / split / "images"
    label_dir = trash_main_path / split / "labels"

    if not image_dir.exists():
        print(f"\n{split.upper()}: Missing")
        continue

    images = [
        p for p in image_dir.iterdir()
        if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
    ]

    image_count = 0
    images_with_trash = 0
    trash_instances = 0
    instance_distribution = Counter()
    annotation_lengths = Counter()
    empty_labels = 0
    missing_labels = 0
    other_class_counts = Counter()

    for image in images:

        image_count += 1
        label = label_dir / f"{image.stem}.txt"

        if not label.exists():
            missing_labels += 1
            continue

        lines = [
            line.strip()
            for line in label.read_text().splitlines()
            if line.strip()
        ]

        if not lines:
            empty_labels += 1
            continue

        image_trash_count = 0

        for line in lines:

            values = line.split()
            class_id = int(float(values[0]))

            annotation_lengths[len(values)] += 1

            if class_id == TRASH_CLASS_ID:
                image_trash_count += 1
                trash_instances += 1
            else:
                other_class_counts[class_id] += 1

        if image_trash_count:
            images_with_trash += 1
            instance_distribution[image_trash_count] += 1

    print(f"\n{split.upper()}")
    print("-" * 45)
    print("Images:", image_count)
    print("Images containing trash cans:", images_with_trash)
    print("Trash can instances:", trash_instances)
    print("Trash cans per image:", dict(instance_distribution))
    print("Annotation line lengths:", dict(annotation_lengths))
    print("Empty label files:", empty_labels)
    print("Missing label files:", missing_labels)

    print("\nOther class counts:")
    for class_id, count in sorted(other_class_counts.items()):
        print(f"{class_id}: {class_names[class_id]} = {count}")

print("\nInspection completed.")


TRASH CAN CLASS ID: 20

PRIMARY TRASHCAN INSTANCE DISTRIBUTION

TRAIN
---------------------------------------------
Images: 84
Images containing trash cans: 83
Trash can instances: 230
Trash cans per image: {4: 16, 2: 4, 3: 30, 1: 27, 5: 3, 9: 1, 6: 1, 11: 1}
Annotation line lengths: {43: 9, 29: 14, 17: 8, 13: 3, 19: 14, 23: 13, 37: 10, 33: 10, 77: 2, 71: 2, 35: 7, 47: 4, 51: 7, 45: 6, 61: 3, 15: 7, 55: 8, 63: 5, 25: 12, 27: 14, 21: 14, 41: 2, 31: 8, 39: 7, 79: 3, 65: 5, 83: 2, 67: 4, 53: 3, 145: 2, 49: 3, 69: 2, 89: 1, 59: 2, 73: 1, 75: 3, 57: 2, 107: 1, 109: 1, 121: 1, 153: 2, 173: 1, 187: 1, 11: 1}
Empty label files: 1
Missing label files: 0

Other class counts:

VALID
---------------------------------------------
Images: 27
Images containing trash cans: 27
Trash can instances: 76
Trash cans per image: {4: 2, 1: 7, 6: 2, 3: 11, 2: 4, 8: 1}
Annotation line lengths: {61: 1, 77: 2, 55: 2, 63: 2, 27: 8, 21: 4, 39: 1, 41: 5, 35: 1, 37: 2, 31: 3, 15: 3, 11: 1, 13: 3, 29: 3, 25: 6, 19: 5, 

In [26]:

from collections import Counter

print("=" * 65)
print("TRASH CAN CLASS-SPECIFIC FORMAT INSPECTION")
print("=" * 65)

for split in ["train", "valid", "test"]:

    label_dir = trash_main_path / split / "labels"

    format_counts = Counter()
    samples = []

    for label_file in sorted(label_dir.glob("*.txt")):

        for line in label_file.read_text().splitlines():

            values = line.strip().split()

            if not values:
                continue

            class_id = int(float(values[0]))

            if class_id != TRASH_CLASS_ID:
                continue

            format_counts[len(values)] += 1

            if len(samples) < 5:
                samples.append((label_file.name, line))

    print(f"\n{split.upper()}")
    print("Trash-can annotation line lengths:", dict(format_counts))

    print("\nSample trash-can annotations:")
    for filename, line in samples:
        print(filename, ":", line)

print("\nInspection complete.")


TRASH CAN CLASS-SPECIFIC FORMAT INSPECTION

TRAIN
Trash-can annotation line lengths: {39: 7, 35: 7, 27: 14, 43: 9, 47: 4, 51: 7, 25: 12, 19: 14, 59: 2, 63: 5, 23: 13, 21: 14, 53: 3, 75: 3, 65: 5, 33: 10, 29: 14, 71: 2, 145: 2, 15: 7, 45: 6, 17: 8, 31: 8, 61: 3, 55: 8, 37: 10, 83: 2, 49: 3, 13: 3, 67: 4, 79: 3, 77: 2, 41: 2, 107: 1, 109: 1, 121: 1, 89: 1, 57: 2, 73: 1, 69: 2, 173: 1, 153: 2, 187: 1, 11: 1}

Sample trash-can annotations:
100_jpg.rf.b0c05fca2e39ba2549aee7572db81d22.txt : 20 0.034722221498371335 0.23428571498371334 0.03125 0.3885714283387622 0.055555555374592835 0.4342857149837134 0.0729166661237785 0.5085714283387622 0.10069444462540716 0.6799999999999999 0.11111111074918567 0.8342857149837134 0.125 0.8514285716612379 0.3125 0.8457142850162865 0.3333333338762215 0.8285714283387623 0.34375 0.7771428566775244 0.32986111074918567 0.42285714332247554 0.35763888925081433 0.3314285716612378 0.34722222149837134 0.19428571498371333 0.32986111074918567 0.14857142833876222 0.309027

In [27]:

# ==========================================
# DOWNLOAD SECONDARY DUSTBIN DATASET
# ==========================================

from pathlib import Path

TRASH_SECONDARY_DIR = SOURCE_DIR / "trashcan_secondary"

# Safety check: never overwrite an existing download
if TRASH_SECONDARY_DIR.exists():
    existing_files = list(TRASH_SECONDARY_DIR.iterdir())

    if existing_files:
        raise FileExistsError(
            f"Destination is not empty: {TRASH_SECONDARY_DIR}\n"
            "Stopping to protect existing data."
        )

    # Roboflow may treat an existing empty directory as already downloaded
    TRASH_SECONDARY_DIR.rmdir()

# Load Roboflow project
trash_secondary_project = rf.workspace(
    "ariyans-workspace-bwwhm"
).project(
    "dustbin-dataset-jktdg"
)

print("PROJECT:", trash_secondary_project.id)

# Select Version 1
trash_secondary_version = trash_secondary_project.version(1)

# Download
trash_secondary_dataset = trash_secondary_version.download(
    "yolov8",
    location=str(TRASH_SECONDARY_DIR),
    overwrite=True
)

trash_secondary_path = Path(trash_secondary_dataset.location)

print("\nDOWNLOAD COMPLETE")
print("Location:", trash_secondary_path)
print("Exists:", trash_secondary_path.exists())

# Locate YAML
secondary_yaml_files = list(trash_secondary_path.rglob("*.yaml"))

print("\nYAML FILES:")
for file in secondary_yaml_files:
    print(file)

# Count images and labels
image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

images = [
    p for p in trash_secondary_path.rglob("*")
    if p.is_file() and p.suffix.lower() in image_extensions
]

labels = list(trash_secondary_path.rglob("*.txt"))

print("\nTOTAL IMAGES:", len(images))
print("TOTAL LABEL FILES:", len(labels))


loading Roboflow workspace...
loading Roboflow project...
PROJECT: ariyans-workspace-bwwhm/dustbin-dataset-jktdg



Extracting Dataset Version Zip to /content/drive/MyDrive/Hazard_Detection/raw_datasets/obstacle/source_datasets/trashcan_secondary in yolov8:: 100%|██████████| 821/821 [00:13<00:00, 62.26it/s]



DOWNLOAD COMPLETE
Location: /content/drive/MyDrive/Hazard_Detection/raw_datasets/obstacle/source_datasets/trashcan_secondary
Exists: True

YAML FILES:
/content/drive/MyDrive/Hazard_Detection/raw_datasets/obstacle/source_datasets/trashcan_secondary/data.yaml

TOTAL IMAGES: 408
TOTAL LABEL FILES: 410


In [28]:

# ==========================================
# INSPECT SECONDARY DUSTBIN DATASET
# ==========================================

import yaml

if not secondary_yaml_files:
    raise FileNotFoundError("No YAML file found in secondary dataset.")

SECONDARY_YAML = secondary_yaml_files[0]

with open(SECONDARY_YAML, "r") as f:
    secondary_data = yaml.safe_load(f)

print("=" * 65)
print("SECONDARY DATASET YAML")
print("=" * 65)

print("Number of classes:", secondary_data.get("nc"))
print("Class names:", secondary_data.get("names"))

print("\n" + "=" * 65)
print("SPLIT DISTRIBUTION")
print("=" * 65)

for split in ["train", "valid", "test"]:
    image_dir = trash_secondary_path / split / "images"
    label_dir = trash_secondary_path / split / "labels"

    image_count = (
        sum(
            1 for p in image_dir.iterdir()
            if p.is_file() and p.suffix.lower() in image_extensions
        )
        if image_dir.exists() else 0
    )

    label_count = (
        len(list(label_dir.glob("*.txt")))
        if label_dir.exists() else 0
    )

    print(f"\n{split.upper()}")
    print("Images:", image_count)
    print("Label files:", label_count)

print("\nYAML CONTENT:")
print(SECONDARY_YAML.read_text())


SECONDARY DATASET YAML
Number of classes: 1
Class names: ['dustbin']

SPLIT DISTRIBUTION

TRAIN
Images: 378
Label files: 378

VALID
Images: 15
Label files: 15

TEST
Images: 15
Label files: 15

YAML CONTENT:
names:
- dustbin
nc: 1
roboflow:
  license: CC BY 4.0
  project: dustbin-dataset-jktdg
  url: https://universe.roboflow.com/ariyans-workspace-bwwhm/dustbin-dataset-jktdg/dataset/1
  version: 1
  workspace: ariyans-workspace-bwwhm
test: ../test/images
train: ../train/images
val: ../valid/images



In [29]:

# ==========================================
# INSPECT SECONDARY DUSTBIN ANNOTATIONS
# ==========================================

from collections import Counter
from pathlib import Path

print("=" * 70)
print("SECONDARY DUSTBIN ANNOTATION DIAGNOSTIC")
print("=" * 70)

for split in ["train", "valid", "test"]:

    image_dir = trash_secondary_path / split / "images"
    label_dir = trash_secondary_path / split / "labels"

    image_files = [
        p for p in image_dir.iterdir()
        if p.is_file() and p.suffix.lower() in image_extensions
    ]

    label_files = list(label_dir.glob("*.txt"))

    instance_count = 0
    images_with_dustbins = 0
    instances_per_image = Counter()
    annotation_lengths = Counter()
    samples = []

    empty_labels = 0
    malformed_lines = 0
    other_class_ids = Counter()

    # Check every image and its corresponding label
    for image_path in image_files:

        label_path = label_dir / f"{image_path.stem}.txt"

        if not label_path.exists():
            continue

        lines = label_path.read_text().splitlines()
        image_instances = 0

        if not lines or not any(line.strip() for line in lines):
            empty_labels += 1

        for line in lines:
            values = line.strip().split()

            if not values:
                continue

            try:
                class_id = int(float(values[0]))
                coordinates = [float(x) for x in values[1:]]

                if class_id != 0:
                    other_class_ids[class_id] += 1
                    continue

                # YOLO bbox = class + 4 coordinates
                # YOLO polygon = class + 6 or more coordinates
                if len(coordinates) < 4 or len(coordinates) % 2 != 0:
                    malformed_lines += 1
                    continue

                annotation_lengths[len(values)] += 1
                image_instances += 1
                instance_count += 1

                if len(samples) < 5:
                    samples.append((label_path.name, line))

            except (ValueError, IndexError):
                malformed_lines += 1

        if image_instances > 0:
            images_with_dustbins += 1
            instances_per_image[image_instances] += 1

    # Labels without corresponding images
    image_stems = {p.stem for p in image_files}
    orphan_labels = [
        p for p in label_files
        if p.stem not in image_stems
    ]

    # Images without corresponding labels
    missing_labels = [
        p for p in image_files
        if not (label_dir / f"{p.stem}.txt").exists()
    ]

    print("\n" + "=" * 60)
    print(split.upper())
    print("=" * 60)

    print("Images:", len(image_files))
    print("Label files:", len(label_files))
    print("Images containing dustbins:", images_with_dustbins)
    print("Dustbin instances:", instance_count)

    print("Dustbins per image:", dict(instances_per_image))
    print("Annotation line lengths:", dict(annotation_lengths))

    print("Empty labels:", empty_labels)
    print("Missing labels:", len(missing_labels))
    print("Orphan labels:", len(orphan_labels))
    print("Malformed lines:", malformed_lines)
    print("Unexpected class IDs:", dict(other_class_ids))

    print("\nSample annotations:")
    for filename, line in samples:
        print(filename, ":", line)

print("\n" + "=" * 70)
print("INSPECTION COMPLETE")
print("=" * 70)


SECONDARY DUSTBIN ANNOTATION DIAGNOSTIC

TRAIN
Images: 378
Label files: 378
Images containing dustbins: 378
Dustbin instances: 378
Dustbins per image: {1: 378}
Annotation line lengths: {5: 378}
Empty labels: 0
Missing labels: 0
Orphan labels: 0
Malformed lines: 0
Unexpected class IDs: {}

Sample annotations:
photo_2_2026-04-07_15-16-41_jpg.rf.972db5d4a5a60a4ec1b7349e2983e231.txt : 0 0.621875 0.60078125 0.75625 0.6296875
IMG_6563_JPG.rf.3f11ef2f780154081a6a82abd64b248b.txt : 0 0.5328125 0.3953125 0.4953125 0.4953125
IMG_6567_JPG.rf.4752a95f6a745f98513512b6024d3fde.txt : 0 0.5171875 0.46328125 0.70234375 0.70078125
IMG_6611_JPG.rf.1384cb724c106ff5f47adb2035ca0dcf.txt : 0 0.5 0.33984375 1 0.58046875
IMG_6564_JPG.rf.504312d1b6c327203369f5a5bf77f5a7.txt : 0 0.421875 0.31640625 0.63671875 0.6140625

VALID
Images: 15
Label files: 15
Images containing dustbins: 15
Dustbin instances: 15
Dustbins per image: {1: 15}
Annotation line lengths: {5: 15}
Empty labels: 0
Missing labels: 0
Orphan labels:

In [30]:

# ==========================================
# COMBINED TRASHCAN INVENTORY
# ==========================================

from collections import Counter
from pathlib import Path
import hashlib

SOURCES = {
    "trashcan_main": {
        "path": trash_main_path,
        "class_id": TRASH_CLASS_ID
    },
    "trashcan_secondary": {
        "path": trash_secondary_path,
        "class_id": 0
    }
}

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


all_records = []
hash_locations = {}

for source_name, source_info in SOURCES.items():

    source_path = source_info["path"]
    class_id = source_info["class_id"]

    print("\n" + "=" * 65)
    print("SOURCE:", source_name)
    print("=" * 65)

    for split in ["train", "valid", "test"]:

        image_dir = source_path / split / "images"
        label_dir = source_path / split / "labels"

        distribution = Counter()
        total_instances = 0
        image_count = 0

        for image_path in sorted(image_dir.iterdir()):

            if not image_path.is_file():
                continue

            if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
                continue

            label_path = label_dir / f"{image_path.stem}.txt"

            if not label_path.exists():
                raise FileNotFoundError(label_path)

            count = 0

            for line in label_path.read_text().splitlines():

                values = line.strip().split()

                if not values:
                    continue

                if int(float(values[0])) == class_id:
                    count += 1

            if count == 0:
                continue

            image_count += 1
            total_instances += count
            distribution[count] += 1

            image_hash = sha256_file(image_path)

            record = {
                "source": source_name,
                "original_split": split,
                "image_path": image_path,
                "label_path": label_path,
                "instance_count": count,
                "sha256": image_hash
            }

            all_records.append(record)

            hash_locations.setdefault(image_hash, []).append(
                (source_name, split, image_path.name)
            )

        print(f"\n{split.upper()}")
        print("Images containing target:", image_count)
        print("Instances:", total_instances)
        print("Instances per image:", dict(sorted(distribution.items())))

# Duplicate image check across both datasets
duplicates = {
    image_hash: locations
    for image_hash, locations in hash_locations.items()
    if len(locations) > 1
}

print("\n" + "=" * 65)
print("COMBINED SUMMARY")
print("=" * 65)

print("Total eligible images:", len(all_records))
print(
    "Total instances:",
    sum(r["instance_count"] for r in all_records)
)

print("Unique image hashes:", len(hash_locations))
print("Duplicate image groups:", len(duplicates))

if duplicates:
    print("\nDUPLICATE GROUPS:")
    for locations in duplicates.values():
        for location in locations:
            print(location)
        print("---")

print("\nInventory complete. No files were modified.")



SOURCE: trashcan_main

TRAIN
Images containing target: 83
Instances: 230
Instances per image: {1: 27, 2: 4, 3: 30, 4: 16, 5: 3, 6: 1, 9: 1, 11: 1}

VALID
Images containing target: 27
Instances: 76
Instances per image: {1: 7, 2: 4, 3: 11, 4: 2, 6: 2, 8: 1}

TEST
Images containing target: 8
Instances: 17
Instances per image: {1: 3, 2: 1, 3: 4}

SOURCE: trashcan_secondary

TRAIN
Images containing target: 378
Instances: 378
Instances per image: {1: 378}

VALID
Images containing target: 15
Instances: 15
Instances per image: {1: 15}

TEST
Images containing target: 15
Instances: 15
Instances per image: {1: 15}

COMBINED SUMMARY
Total eligible images: 526
Total instances: 731
Unique image hashes: 525
Duplicate image groups: 1

DUPLICATE GROUPS:
('trashcan_secondary', 'train', 'IMG_6604_JPG.rf.0ca8abb92b63e2b75d2965c9b3e84765.jpg')
('trashcan_secondary', 'train', 'IMG_6604_JPG.rf.2980fedf8d5d1fc5c18104ccce0b4398.jpg')
---

Inventory complete. No files were modified.


This checks it, doesnt alter the memory. If it passes, we will actually split it

In [31]:

# ==========================================
# CREATE TRASHCAN SELECTION PLAN
# ==========================================

import random
from collections import Counter, defaultdict

SEED = 42
rng = random.Random(SEED)

TARGETS = {
    "train": 220,
    "valid": 80,
    "test": 100
}

# ------------------------------------------
# 1. Remove exact duplicate images
# ------------------------------------------

unique_records = []
seen_hashes = set()
duplicate_records = []

for record in all_records:

    if record["sha256"] in seen_hashes:
        duplicate_records.append(record)
        continue

    seen_hashes.add(record["sha256"])
    unique_records.append(record)

print("Original eligible images:", len(all_records))
print("Exact duplicate copies excluded:", len(duplicate_records))
print("Unique eligible images:", len(unique_records))

# ------------------------------------------
# 2. Organize source pools
# ------------------------------------------

pools = defaultdict(list)

for record in unique_records:
    key = (record["source"], record["original_split"])
    pools[key].append(record)

# ------------------------------------------
# 3. Select an exact number of instances
#    using complete images only
# ------------------------------------------

def select_exact_instances(records, target, seed):

    shuffled = records.copy()
    random.Random(seed).shuffle(shuffled)

    # Dynamic programming: total instances -> selected records
    dp = {0: []}

    for record in shuffled:

        count = record["instance_count"]

        for current_total in sorted(list(dp.keys()), reverse=True):

            new_total = current_total + count

            if new_total <= target and new_total not in dp:
                dp[new_total] = dp[current_total] + [record]

        if target in dp:
            return dp[target]

    raise ValueError(
        f"Cannot reach exactly {target} instances "
        f"using the available complete images."
    )

# ------------------------------------------
# 4. TRAIN: 100 primary + 120 secondary
# ------------------------------------------

main_train = select_exact_instances(
    pools[("trashcan_main", "train")],
    target=100,
    seed=SEED
)

secondary_train_pool = pools[
    ("trashcan_secondary", "train")
].copy()

rng.shuffle(secondary_train_pool)

secondary_train = secondary_train_pool[:120]

remaining_secondary_train = secondary_train_pool[120:]

# ------------------------------------------
# 5. VALID: 76 primary + 4 secondary
# ------------------------------------------

main_valid = pools[
    ("trashcan_main", "valid")
].copy()

if sum(r["instance_count"] for r in main_valid) != 76:
    raise ValueError("Unexpected primary validation count.")

secondary_valid_pool = pools[
    ("trashcan_secondary", "valid")
].copy()

secondary_valid = rng.sample(secondary_valid_pool, 4)

# ------------------------------------------
# 6. TEST: 17 primary + 15 secondary
#    original test + 68 secondary train
# ------------------------------------------

main_test = pools[
    ("trashcan_main", "test")
].copy()

secondary_test = pools[
    ("trashcan_secondary", "test")
].copy()

secondary_train_for_test = remaining_secondary_train[:68]

# ------------------------------------------
# 7. Assemble final selection
# ------------------------------------------

TRASH_SELECTION = {
    "train": main_train + secondary_train,

    "valid": main_valid + secondary_valid,

    "test": (
        main_test
        + secondary_test
        + secondary_train_for_test
    )
}

# ------------------------------------------
# 8. Verify instance counts
# ------------------------------------------

print("\n" + "=" * 70)
print("FINAL SELECTION COUNTS")
print("=" * 70)

for split, records in TRASH_SELECTION.items():

    counts = Counter()
    total = 0

    for record in records:
        counts[record["source"]] += record["instance_count"]
        total += record["instance_count"]

    print(f"\n{split.upper()}")
    print("Images selected:", len(records))
    print("Primary instances:", counts["trashcan_main"])
    print("Secondary instances:", counts["trashcan_secondary"])
    print("Total instances:", total)
    print("Target:", TARGETS[split])

    assert total == TARGETS[split], (
        f"{split} count does not match target!"
    )

# ------------------------------------------
# 9. Verify no image leakage across splits
# ------------------------------------------

split_hashes = {}

for split, records in TRASH_SELECTION.items():

    hashes = [r["sha256"] for r in records]

    if len(hashes) != len(set(hashes)):
        raise ValueError(f"Duplicate images inside {split}!")

    split_hashes[split] = set(hashes)

assert split_hashes["train"].isdisjoint(split_hashes["valid"])
assert split_hashes["train"].isdisjoint(split_hashes["test"])
assert split_hashes["valid"].isdisjoint(split_hashes["test"])

print("\nNo duplicate images across final splits.")

# ------------------------------------------
# 10. Show original split provenance
# ------------------------------------------

print("\n" + "=" * 70)
print("SOURCE SPLIT PROVENANCE")
print("=" * 70)

for final_split, records in TRASH_SELECTION.items():

    provenance = Counter(
        (r["source"], r["original_split"])
        for r in records
    )

    print(f"\n{final_split.upper()}")

    for (source, original_split), image_count in sorted(
        provenance.items()
    ):
        print(
            f"{source:22s} | "
            f"{original_split:5s} | "
            f"{image_count} images"
        )

print("\n" + "=" * 70)
print("SELECTION PLAN CREATED SUCCESSFULLY")
print("No source or final dataset files were modified.")
print("=" * 70)


Original eligible images: 526
Exact duplicate copies excluded: 1
Unique eligible images: 525

FINAL SELECTION COUNTS

TRAIN
Images selected: 152
Primary instances: 100
Secondary instances: 120
Total instances: 220
Target: 220

VALID
Images selected: 31
Primary instances: 76
Secondary instances: 4
Total instances: 80
Target: 80

TEST
Images selected: 91
Primary instances: 17
Secondary instances: 83
Total instances: 100
Target: 100

No duplicate images across final splits.

SOURCE SPLIT PROVENANCE

TRAIN
trashcan_main          | train | 32 images
trashcan_secondary     | train | 120 images

VALID
trashcan_main          | valid | 27 images
trashcan_secondary     | valid | 4 images

TEST
trashcan_main          | test  | 8 images
trashcan_secondary     | test  | 15 images
trashcan_secondary     | train | 68 images

SELECTION PLAN CREATED SUCCESSFULLY
No source or final dataset files were modified.


We'll now integrate the selected images into final_dataset. The primary source's polygons will be converted into enclosing bounding boxes; the secondary source's existing bounding boxes will be retained unchanged geometrically. Both will be remapped to class 13.

In [32]:

# ==========================================
# INTEGRATE TRASHCAN DATASETS INTO FINAL
# ==========================================

import json
import shutil
import pandas as pd
from pathlib import Path

# ------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------

FINAL_SPLITS = ["train", "valid", "test"]

EXPECTED_COLUMNS = [
    "image_filename",
    "source_dataset",
    "source_workspace",
    "source_project",
    "source_version",
    "source_split",
    "final_split",
    "original_class_counts",
    "original_instance_count",
    "final_class_id",
    "final_instance_count",
    "polygon_conversions"
]

FINAL_CLASS_ID = 13

SOURCE_METADATA = {
    "trashcan_main": {
        "workspace": "group10textdetect",
        "project": "trashcan-detect",
        "version": 6,
        "class_name": "trash can"
    },
    "trashcan_secondary": {
        "workspace": "ariyans-workspace-bwwhm",
        "project": "dustbin-dataset-jktdg",
        "version": 1,
        "class_name": "dustbin"
    }
}

# ------------------------------------------
# 2. VERIFY FINAL DIRECTORY STRUCTURE
# ------------------------------------------

for split in FINAL_SPLITS:

    for folder in ["images", "labels"]:

        path = FINAL / split / folder
        path.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = FINAL / "collection_manifest.csv"

if MANIFEST_PATH.exists():

    existing_manifest = pd.read_csv(MANIFEST_PATH)

    if list(existing_manifest.columns) != EXPECTED_COLUMNS:
        raise ValueError(
            "Existing manifest columns do not match expected schema.\n"
            f"Found: {list(existing_manifest.columns)}"
        )

else:
    existing_manifest = pd.DataFrame(columns=EXPECTED_COLUMNS)

# ------------------------------------------
# 3. PREPARE AND VALIDATE ALL ANNOTATIONS
#    WITHOUT COPYING ANY FILES
# ------------------------------------------

prepared_records = []
new_manifest_rows = []

for final_split, records in TRASH_SELECTION.items():

    for record in records:

        source_name = record["source"]
        metadata = SOURCE_METADATA[source_name]

        image_path = record["image_path"]
        label_path = record["label_path"]

        # Prefix filenames to prevent source collisions
        new_filename = (
            f"{source_name}__{image_path.name}"
        )

        new_label_filename = (
            f"{Path(new_filename).stem}.txt"
        )

        destination_image = (
            FINAL / final_split / "images" / new_filename
        )

        destination_label = (
            FINAL / final_split / "labels" / new_label_filename
        )

        # Safety: never overwrite existing files
        if destination_image.exists():
            raise FileExistsError(destination_image)

        if destination_label.exists():
            raise FileExistsError(destination_label)

        if new_filename in set(
            existing_manifest["image_filename"].astype(str)
        ):
            raise ValueError(
                f"Filename already exists in manifest: {new_filename}"
            )

        # Read and remap annotations
        output_lines = []
        polygon_count = 0
        original_instance_count = 0

        for line in label_path.read_text().splitlines():

            values = line.strip().split()

            if not values:
                continue

            original_class_id = int(float(values[0]))

            expected_class_id = (
                SOURCES[source_name]["class_id"]
            )

            # Ignore all non-target classes
            if original_class_id != expected_class_id:
                continue

            coordinates = [
                float(value) for value in values[1:]
            ]

            # ----------------------------------
            # Existing bounding box
            # ----------------------------------

            if len(coordinates) == 4:

                xc, yc, width, height = coordinates

                if width <= 0 or height <= 0:
                    raise ValueError(
                        f"Invalid bounding box: {label_path}"
                    )

                if not all(0 <= x <= 1 for x in coordinates):
                    raise ValueError(
                        f"Out-of-range bbox coordinates: {label_path}"
                    )

                if (
                    xc - width / 2 < -1e-6
                    or yc - height / 2 < -1e-6
                    or xc + width / 2 > 1.000001
                    or yc + height / 2 > 1.000001
                ):
                    raise ValueError(
                        f"Bounding box exceeds image boundaries: {label_path}"
                    )

                output_lines.append(
                    f"{FINAL_CLASS_ID} "
                    + " ".join(values[1:])
                )

            # ----------------------------------
            # Polygon -> enclosing bounding box
            # ----------------------------------

            elif len(coordinates) >= 6 and len(coordinates) % 2 == 0:

                xs = coordinates[0::2]
                ys = coordinates[1::2]

                if not all(
                    0 <= x <= 1 for x in xs
                ) or not all(
                    0 <= y <= 1 for y in ys
                ):
                    raise ValueError(
                        f"Out-of-range polygon coordinates: {label_path}"
                    )

                x_min = min(xs)
                x_max = max(xs)
                y_min = min(ys)
                y_max = max(ys)

                xc = (x_min + x_max) / 2
                yc = (y_min + y_max) / 2
                width = x_max - x_min
                height = y_max - y_min

                if width <= 0 or height <= 0:
                    raise ValueError(
                        f"Degenerate polygon: {label_path}"
                    )

                output_lines.append(
                    f"{FINAL_CLASS_ID} "
                    f"{xc:.8f} {yc:.8f} "
                    f"{width:.8f} {height:.8f}"
                )

                polygon_count += 1

            else:
                raise ValueError(
                    f"Unexpected annotation format in {label_path}: "
                    f"{len(coordinates) + 1} values"
                )

            original_instance_count += 1

        if original_instance_count != record["instance_count"]:
            raise ValueError(
                f"Instance mismatch for {image_path.name}: "
                f"expected {record['instance_count']}, "
                f"found {original_instance_count}"
            )

        if not output_lines:
            raise ValueError(
                f"No target annotations found: {label_path}"
            )

        prepared_records.append({
            "source_record": record,
            "final_split": final_split,
            "new_filename": new_filename,
            "new_label_filename": new_label_filename,
            "destination_image": destination_image,
            "destination_label": destination_label,
            "output_lines": output_lines
        })

        new_manifest_rows.append({
            "image_filename": new_filename,
            "source_dataset": source_name,
            "source_workspace": metadata["workspace"],
            "source_project": metadata["project"],
            "source_version": metadata["version"],
            "source_split": record["original_split"],
            "final_split": final_split,
            "original_class_counts": json.dumps({
                metadata["class_name"]: original_instance_count
            }),
            "original_instance_count": original_instance_count,
            "final_class_id": FINAL_CLASS_ID,
            "final_instance_count": len(output_lines),
            "polygon_conversions": polygon_count
        })

# ------------------------------------------
# 4. FINAL PREFLIGHT CHECK
# ------------------------------------------

new_filenames = [
    row["image_filename"] for row in new_manifest_rows
]

if len(new_filenames) != len(set(new_filenames)):
    raise ValueError("Duplicate output filenames detected.")

if set(new_filenames) & set(
    existing_manifest["image_filename"].astype(str)
):
    raise ValueError("Output filenames collide with existing manifest.")

print("=" * 65)
print("PREFLIGHT PASSED")
print("=" * 65)

print("Images prepared:", len(prepared_records))
print("Manifest rows prepared:", len(new_manifest_rows))

print(
    "Polygon conversions:",
    sum(r["polygon_conversions"] for r in new_manifest_rows)
)

print(
    "Existing manifest rows preserved:",
    len(existing_manifest)
)

# ------------------------------------------
# 5. COPY IMAGES AND WRITE LABELS
# ------------------------------------------

for item in prepared_records:

    shutil.copy2(
        item["source_record"]["image_path"],
        item["destination_image"]
    )

    item["destination_label"].write_text(
        "\n".join(item["output_lines"]) + "\n"
    )

# ------------------------------------------
# 6. APPEND MANIFEST
# ------------------------------------------

new_manifest = pd.DataFrame(
    new_manifest_rows,
    columns=EXPECTED_COLUMNS
)

combined_manifest = pd.concat(
    [existing_manifest, new_manifest],
    ignore_index=True
)

combined_manifest.to_csv(
    MANIFEST_PATH,
    index=False
)

# ------------------------------------------
# 7. REPORT
# ------------------------------------------

print("\n" + "=" * 65)
print("TRASHCAN INTEGRATION COMPLETE")
print("=" * 65)

for split in FINAL_SPLITS:

    split_rows = new_manifest[
        new_manifest["final_split"] == split
    ]

    print(f"\n{split.upper()}")
    print("Images added:", len(split_rows))
    print(
        "Instances added:",
        split_rows["final_instance_count"].sum()
    )
    print(
        "Polygon conversions:",
        split_rows["polygon_conversions"].sum()
    )

print("\nManifest:", MANIFEST_PATH)
print("Total manifest rows:", len(combined_manifest))


PREFLIGHT PASSED
Images prepared: 274
Manifest rows prepared: 274
Polygon conversions: 193
Existing manifest rows preserved: 512

TRASHCAN INTEGRATION COMPLETE

TRAIN
Images added: 152
Instances added: 220
Polygon conversions: 100

VALID
Images added: 31
Instances added: 80
Polygon conversions: 76

TEST
Images added: 91
Instances added: 100
Polygon conversions: 17

Manifest: /content/drive/MyDrive/Hazard_Detection/raw_datasets/obstacle/final_dataset/collection_manifest.csv
Total manifest rows: 786


The final verification of dataset

In [34]:

# ==========================================
# VERIFY TRASHCAN INTEGRATION (FIXED)
# ==========================================

import pandas as pd
from pathlib import Path

manifest = pd.read_csv(MANIFEST_PATH)

trash_manifest = manifest[
    manifest["source_dataset"].isin([
        "trashcan_main",
        "trashcan_secondary"
    ])
]

print("=" * 65)
print("TRASHCAN MANIFEST VERIFICATION")
print("=" * 65)

for split in ["train", "valid", "test"]:

    split_rows = trash_manifest[
        trash_manifest["final_split"] == split
    ]

    print(f"\n{split.upper()}")
    print("Images:", len(split_rows))
    print(
        "Instances:",
        split_rows["final_instance_count"].sum()
    )
    print(
        "Polygon conversions:",
        split_rows["polygon_conversions"].sum()
    )

print("\n" + "=" * 65)
print("FILE AND LABEL VALIDATION")
print("=" * 65)

for split in ["train", "valid", "test"]:

    image_dir = FINAL / split / "images"
    label_dir = FINAL / split / "labels"

    split_rows = trash_manifest[
        trash_manifest["final_split"] == split
    ]

    trash_images = set(
        split_rows["image_filename"].astype(str).tolist()
    )

    missing_images = []
    missing_labels = []
    wrong_class = 0
    malformed = 0
    actual_instances = 0

    for filename in trash_images:

        image_path = image_dir / filename
        label_path = label_dir / f"{Path(filename).stem}.txt"

        if not image_path.exists():
            missing_images.append(filename)

        if not label_path.exists():
            missing_labels.append(label_path.name)
            continue

        for line in label_path.read_text().splitlines():

            values = line.strip().split()

            if not values:
                continue

            if len(values) != 5:
                malformed += 1
                continue

            try:
                class_id = int(values[0])
                coords = [float(v) for v in values[1:]]

                if class_id != 13:
                    wrong_class += 1
                    continue

                xc, yc, width, height = coords

                if not all(0 <= v <= 1 for v in coords):
                    malformed += 1
                    continue

                if width <= 0 or height <= 0:
                    malformed += 1
                    continue

                if (
                    xc - width / 2 < -1e-6
                    or yc - height / 2 < -1e-6
                    or xc + width / 2 > 1.000001
                    or yc + height / 2 > 1.000001
                ):
                    malformed += 1
                    continue

                actual_instances += 1

            except ValueError:
                malformed += 1

    expected_instances = int(
        split_rows["final_instance_count"].sum()
    )

    print(f"\n{split.upper()}")
    print("Expected images:", len(trash_images))
    print("Missing images:", len(missing_images))
    print("Missing labels:", len(missing_labels))
    print("Wrong class IDs:", wrong_class)
    print("Malformed annotations:", malformed)
    print("Expected instances:", expected_instances)
    print("Actual instances:", actual_instances)

    assert not missing_images
    assert not missing_labels
    assert wrong_class == 0
    assert malformed == 0
    assert actual_instances == expected_instances

print("\n" + "=" * 65)
print("ALL TRASHCAN CHECKS PASSED")
print("=" * 65)


TRASHCAN MANIFEST VERIFICATION

TRAIN
Images: 152
Instances: 220
Polygon conversions: 100

VALID
Images: 31
Instances: 80
Polygon conversions: 76

TEST
Images: 91
Instances: 100
Polygon conversions: 17

FILE AND LABEL VALIDATION

TRAIN
Expected images: 152
Missing images: 0
Missing labels: 0
Wrong class IDs: 0
Malformed annotations: 0
Expected instances: 220
Actual instances: 220

VALID
Expected images: 31
Missing images: 0
Missing labels: 0
Wrong class IDs: 0
Malformed annotations: 0
Expected instances: 80
Actual instances: 80

TEST
Expected images: 91
Missing images: 0
Missing labels: 0
Wrong class IDs: 0
Malformed annotations: 0
Expected instances: 100
Actual instances: 100

ALL TRASHCAN CHECKS PASSED


Collecting the fire hydrant dataset from here

In [35]:

# ==========================================
# FIRE HYDRANTS — COCO 2017 ANNOTATIONS
# ==========================================

from pathlib import Path
import zipfile
import requests
import shutil

# ------------------------------------------
# 1. PATHS
# ------------------------------------------

HYDRANT_SOURCE_DIR = SOURCE_DIR / "fire_hydrants"
COCO_ANNOTATION_DIR = HYDRANT_SOURCE_DIR / "coco_annotations"

HYDRANT_SOURCE_DIR.mkdir(parents=True, exist_ok=True)

if COCO_ANNOTATION_DIR.exists() and any(
    COCO_ANNOTATION_DIR.iterdir()
):
    raise FileExistsError(
        f"Annotation directory is not empty:\n"
        f"{COCO_ANNOTATION_DIR}\n"
        "Stopping to protect existing files."
    )

COCO_ANNOTATION_DIR.mkdir(parents=True, exist_ok=True)

ZIP_PATH = HYDRANT_SOURCE_DIR / "annotations_trainval2017.zip"

# ------------------------------------------
# 2. DOWNLOAD ANNOTATION ARCHIVE
# ------------------------------------------

COCO_URL = (
    "http://images.cocodataset.org/annotations/"
    "annotations_trainval2017.zip"
)

print("Downloading COCO 2017 annotations...")
print("URL:", COCO_URL)

response = requests.get(
    COCO_URL,
    stream=True,
    timeout=(30, 180)
)

response.raise_for_status()

total_bytes = int(response.headers.get("content-length", 0))
downloaded_bytes = 0

with open(ZIP_PATH, "wb") as f:

    for chunk in response.iter_content(chunk_size=1024 * 1024):

        if not chunk:
            continue

        f.write(chunk)
        downloaded_bytes += len(chunk)

        if total_bytes:
            percent = downloaded_bytes / total_bytes * 100
            print(
                f"\rDownloaded: {percent:.1f}%",
                end=""
            )

print("\nDownload complete.")
print("Archive size (MB):", round(ZIP_PATH.stat().st_size / (1024**2), 2))

# ------------------------------------------
# 3. EXTRACT ONLY REQUIRED JSON FILES
# ------------------------------------------

REQUIRED_FILES = {
    "annotations/instances_train2017.json",
    "annotations/instances_val2017.json"
}

with zipfile.ZipFile(ZIP_PATH, "r") as archive:

    available = set(archive.namelist())

    missing = REQUIRED_FILES - available

    if missing:
        raise FileNotFoundError(
            f"Missing expected annotation files: {missing}"
        )

    for member in REQUIRED_FILES:

        destination = COCO_ANNOTATION_DIR / Path(member).name

        with archive.open(member) as source, open(destination, "wb") as target:
            shutil.copyfileobj(source, target)

        print("Extracted:", destination.name)

# ------------------------------------------
# 4. VERIFY
# ------------------------------------------

train_json = COCO_ANNOTATION_DIR / "instances_train2017.json"
val_json = COCO_ANNOTATION_DIR / "instances_val2017.json"

print("\n" + "=" * 65)
print("ANNOTATION DOWNLOAD VERIFICATION")
print("=" * 65)

print("Train annotations exist:", train_json.exists())
print("Validation annotations exist:", val_json.exists())

print(
    "Train JSON size (MB):",
    round(train_json.stat().st_size / (1024**2), 2)
)

print(
    "Validation JSON size (MB):",
    round(val_json.stat().st_size / (1024**2), 2)
)

print("\nAnnotation files ready.")


URL: http://images.cocodataset.org/annotations/annotations_trainval2017.zip
Downloaded: 100.0%
Download complete.
Archive size (MB): 241.19
Extracted: instances_train2017.json
Extracted: instances_val2017.json

ANNOTATION DOWNLOAD VERIFICATION
Train annotations exist: True
Validation annotations exist: True
Train JSON size (MB): 448.02
Validation JSON size (MB): 19.06

Annotation files ready.


In [36]:

# ==========================================
# COCO 2017 — FIRE HYDRANT INVENTORY
# ==========================================

import json
from collections import Counter, defaultdict
from pathlib import Path

TRAIN_JSON = COCO_ANNOTATION_DIR / "instances_train2017.json"
VAL_JSON = COCO_ANNOTATION_DIR / "instances_val2017.json"

# ------------------------------------------
# 1. LOAD ANNOTATIONS
# ------------------------------------------

with open(TRAIN_JSON, "r") as f:
    coco_train = json.load(f)

with open(VAL_JSON, "r") as f:
    coco_val = json.load(f)

print("COCO JSON files loaded successfully.")

# ------------------------------------------
# 2. FIND FIRE HYDRANT CATEGORY
# ------------------------------------------

categories = coco_train["categories"]

hydrant_categories = [
    category for category in categories
    if category["name"].lower() == "fire hydrant"
]

if len(hydrant_categories) != 1:
    raise ValueError(
        f"Expected exactly one fire hydrant category. "
        f"Found: {hydrant_categories}"
    )

HYDRANT_COCO_CATEGORY_ID = hydrant_categories[0]["id"]

print("\nFire hydrant category:")
print(hydrant_categories[0])

# ------------------------------------------
# 3. INSPECT EACH COCO SPLIT
# ------------------------------------------

def inspect_coco_hydrants(coco_data, split_name):

    images_by_id = {
        image["id"]: image
        for image in coco_data["images"]
    }

    hydrant_annotations = [
        ann for ann in coco_data["annotations"]
        if ann["category_id"] == HYDRANT_COCO_CATEGORY_ID
    ]

    annotations_by_image = defaultdict(list)

    for ann in hydrant_annotations:
        annotations_by_image[ann["image_id"]].append(ann)

    instance_distribution = Counter(
        len(annotations)
        for annotations in annotations_by_image.values()
    )

    print("\n" + "=" * 65)
    print(split_name.upper())
    print("=" * 65)

    print("Total COCO images:", len(coco_data["images"]))
    print("Fire hydrant instances:", len(hydrant_annotations))
    print("Images containing fire hydrants:", len(annotations_by_image))
    print("Hydrants per image:", dict(sorted(instance_distribution.items())))

    missing_images = [
        image_id for image_id in annotations_by_image
        if image_id not in images_by_id
    ]

    print("Annotations with missing image metadata:", len(missing_images))

    # Check bounding-box availability and validity
    valid_bbox_count = 0
    invalid_bbox_count = 0

    for ann in hydrant_annotations:

        bbox = ann.get("bbox")

        if (
            isinstance(bbox, list)
            and len(bbox) == 4
            and all(isinstance(x, (int, float)) for x in bbox)
            and bbox[2] > 0
            and bbox[3] > 0
        ):
            valid_bbox_count += 1
        else:
            invalid_bbox_count += 1

    print("Valid COCO bounding boxes:", valid_bbox_count)
    print("Invalid bounding boxes:", invalid_bbox_count)

    # Save records for later selection
    records = []

    for image_id, annotations in annotations_by_image.items():

        if image_id not in images_by_id:
            continue

        records.append({
            "image_id": image_id,
            "image_info": images_by_id[image_id],
            "annotations": annotations,
            "instance_count": len(annotations),
            "original_split": split_name
        })

    return records


COCO_TRAIN_HYDRANTS = inspect_coco_hydrants(
    coco_train, "train"
)

COCO_VAL_HYDRANTS = inspect_coco_hydrants(
    coco_val, "valid"
)

# ------------------------------------------
# 4. COMBINED SUMMARY
# ------------------------------------------

print("\n" + "=" * 65)
print("COMBINED FIRE HYDRANT INVENTORY")
print("=" * 65)

print(
    "Total instances:",
    sum(r["instance_count"] for r in COCO_TRAIN_HYDRANTS)
    + sum(r["instance_count"] for r in COCO_VAL_HYDRANTS)
)

print(
    "Total eligible images:",
    len(COCO_TRAIN_HYDRANTS) + len(COCO_VAL_HYDRANTS)
)

print("\nInventory complete. No images downloaded.")


COCO JSON files loaded successfully.

Fire hydrant category:
{'supercategory': 'outdoor', 'id': 11, 'name': 'fire hydrant'}

TRAIN
Total COCO images: 118287
Fire hydrant instances: 1865
Images containing fire hydrants: 1711
Hydrants per image: {1: 1629, 2: 55, 3: 11, 4: 8, 5: 2, 6: 1, 7: 1, 8: 1, 9: 1, 10: 1, 11: 1}
Annotations with missing image metadata: 0
Valid COCO bounding boxes: 1865
Invalid bounding boxes: 0

VALID
Total COCO images: 5000
Fire hydrant instances: 101
Images containing fire hydrants: 86
Hydrants per image: {1: 79, 2: 5, 3: 1, 9: 1}
Annotations with missing image metadata: 0
Valid COCO bounding boxes: 101
Invalid bounding boxes: 0

COMBINED FIRE HYDRANT INVENTORY
Total instances: 1966
Total eligible images: 1797

Inventory complete. No images downloaded.


In [37]:

# ==========================================
# COCO FIRE HYDRANTS — EXACT IMAGE SELECTION
# ==========================================

import random
from collections import Counter

SEED = 42

HYDRANT_TARGETS = {
    "train": 220,
    "valid": 80,
    "test": 100
}

# ------------------------------------------
# 1. SELECT COMPLETE IMAGES
# ------------------------------------------

def select_exact_images(records, target, seed):

    shuffled = records.copy()
    random.Random(seed).shuffle(shuffled)

    # Dynamic programming:
    # total instances -> selected image records
    dp = {0: []}

    for record in shuffled:

        count = record["instance_count"]

        for current_total in sorted(
            list(dp.keys()), reverse=True
        ):

            new_total = current_total + count

            if new_total <= target and new_total not in dp:
                dp[new_total] = dp[current_total] + [record]

        if target in dp:
            return dp[target]

    available = sum(r["instance_count"] for r in records)

    raise ValueError(
        f"Cannot reach exactly {target} instances. "
        f"Available: {available}"
    )


# ------------------------------------------
# 2. TRAIN SELECTION
# ------------------------------------------

hydrant_train_selection = select_exact_images(
    COCO_TRAIN_HYDRANTS,
    target=HYDRANT_TARGETS["train"],
    seed=SEED
)

selected_train_ids = {
    r["image_id"] for r in hydrant_train_selection
}

remaining_train_records = [
    r for r in COCO_TRAIN_HYDRANTS
    if r["image_id"] not in selected_train_ids
]


# ------------------------------------------
# 3. TEST SELECTION
#    From remaining original COCO train images
# ------------------------------------------

hydrant_test_selection = select_exact_images(
    remaining_train_records,
    target=HYDRANT_TARGETS["test"],
    seed=SEED + 1
)

selected_test_ids = {
    r["image_id"] for r in hydrant_test_selection
}


# ------------------------------------------
# 4. VALIDATION SELECTION
#    From original COCO validation images
# ------------------------------------------

hydrant_valid_selection = select_exact_images(
    COCO_VAL_HYDRANTS,
    target=HYDRANT_TARGETS["valid"],
    seed=SEED + 2
)


# ------------------------------------------
# 5. ASSEMBLE FINAL SELECTION
# ------------------------------------------

HYDRANT_SELECTION = {
    "train": hydrant_train_selection,
    "valid": hydrant_valid_selection,
    "test": hydrant_test_selection
}


# ------------------------------------------
# 6. VERIFY COUNTS
# ------------------------------------------

print("=" * 70)
print("COCO FIRE HYDRANT SELECTION")
print("=" * 70)

for split, records in HYDRANT_SELECTION.items():

    total_instances = sum(
        r["instance_count"] for r in records
    )

    print(f"\n{split.upper()}")
    print("Selected images:", len(records))
    print("Selected instances:", total_instances)
    print("Target:", HYDRANT_TARGETS[split])

    assert total_instances == HYDRANT_TARGETS[split]


# ------------------------------------------
# 7. VERIFY IMAGE SEPARATION
# ------------------------------------------

train_ids = {
    r["image_id"] for r in HYDRANT_SELECTION["train"]
}

valid_ids = {
    r["image_id"] for r in HYDRANT_SELECTION["valid"]
}

test_ids = {
    r["image_id"] for r in HYDRANT_SELECTION["test"]
}

assert train_ids.isdisjoint(valid_ids)
assert train_ids.isdisjoint(test_ids)
assert valid_ids.isdisjoint(test_ids)

print("\nNo image IDs overlap across final splits.")


# ------------------------------------------
# 8. SOURCE PROVENANCE
# ------------------------------------------

print("\n" + "=" * 70)
print("ORIGINAL COCO SPLIT PROVENANCE")
print("=" * 70)

for final_split, records in HYDRANT_SELECTION.items():

    provenance = Counter(
        r["original_split"] for r in records
    )

    print(f"\n{final_split.upper()}")

    for original_split, count in provenance.items():
        print(
            f"Original COCO {original_split}: "
            f"{count} images"
        )


# ------------------------------------------
# 9. DOWNLOAD ESTIMATE
# ------------------------------------------

total_selected_images = sum(
    len(records)
    for records in HYDRANT_SELECTION.values()
)

print("\n" + "=" * 70)
print("DOWNLOAD ESTIMATE")
print("=" * 70)

print("Total selected images:", total_selected_images)
print("Total selected annotations:", sum(HYDRANT_TARGETS.values()))
print("Images not selected:", 1797 - total_selected_images)

print("\nSelection complete. No images downloaded.")


COCO FIRE HYDRANT SELECTION

TRAIN
Selected images: 196
Selected instances: 220
Target: 220

VALID
Selected images: 77
Selected instances: 80
Target: 80

TEST
Selected images: 91
Selected instances: 100
Target: 100

No image IDs overlap across final splits.

ORIGINAL COCO SPLIT PROVENANCE

TRAIN
Original COCO train: 196 images

VALID
Original COCO valid: 77 images

TEST
Original COCO train: 91 images

DOWNLOAD ESTIMATE
Total selected images: 364
Total selected annotations: 400
Images not selected: 1433

Selection complete. No images downloaded.


In [38]:

import requests
import time
from pathlib import Path
from PIL import Image
from io import BytesIO

# ==========================================
# CONFIGURATION
# ==========================================

HYDRANT_IMAGE_DIR = HYDRANT_SOURCE_DIR / "selected_images"

# COCO image servers: HTTP avoids the SSL issue
# encountered earlier with images.cocodataset.org.
COCO_IMAGE_URLS = {
    "train": "http://images.cocodataset.org/train2017/",
    "valid": "http://images.cocodataset.org/val2017/"
}

# ==========================================
# SAFETY CHECKS
# ==========================================

if HYDRANT_IMAGE_DIR.exists():
    existing_files = [
        p for p in HYDRANT_IMAGE_DIR.rglob("*")
        if p.is_file()
    ]

    if existing_files:
        raise FileExistsError(
            f"Destination already contains files: {HYDRANT_IMAGE_DIR}\n"
            "Stopping to avoid overwriting existing downloads."
        )
else:
    HYDRANT_IMAGE_DIR.mkdir(parents=True)

# Separate folders preserve original COCO split provenance.
for split in ("train", "valid"):
    (HYDRANT_IMAGE_DIR / split).mkdir(exist_ok=True)

# ==========================================
# BUILD DOWNLOAD LIST
# ==========================================

download_jobs = []

for final_split, records in HYDRANT_SELECTION.items():

    for record in records:

        original_split = record["original_split"]
        image_info = record["image_info"]

        if original_split not in COCO_IMAGE_URLS:
            raise ValueError(
                f"Unexpected original split: {original_split}"
            )

        download_jobs.append({
            "final_split": final_split,
            "original_split": original_split,
            "image_info": image_info,
            "record": record
        })

# Ensure exactly the expected number of unique images.
image_ids = [
    job["image_info"]["id"]
    for job in download_jobs
]

assert len(download_jobs) == 364
assert len(set(image_ids)) == 364

print("=" * 65)
print("COCO FIRE HYDRANT IMAGE DOWNLOAD")
print("=" * 65)
print("Images to download:", len(download_jobs))
print("Expected annotations:", 400)
print("Destination:", HYDRANT_IMAGE_DIR)

# ==========================================
# DOWNLOAD FUNCTION
# ==========================================

def download_coco_image(job, max_retries=3):

    image_info = job["image_info"]
    original_split = job["original_split"]

    filename = image_info["file_name"]

    output_dir = HYDRANT_IMAGE_DIR / original_split
    output_path = output_dir / filename
    temp_path = output_dir / (filename + ".part")

    url = COCO_IMAGE_URLS[original_split] + filename

    for attempt in range(1, max_retries + 1):

        try:
            response = requests.get(
                url,
                stream=True,
                timeout=45
            )

            response.raise_for_status()

            with open(temp_path, "wb") as f:
                for chunk in response.iter_content(
                    chunk_size=1024 * 64
                ):
                    if chunk:
                        f.write(chunk)

            # Verify that the downloaded file is a valid image.
            with Image.open(temp_path) as img:
                img.verify()

            # Verify expected dimensions from COCO metadata.
            with Image.open(temp_path) as img:
                if img.size != (
                    image_info["width"],
                    image_info["height"]
                ):
                    raise ValueError(
                        f"Dimension mismatch for {filename}: "
                        f"expected {(image_info['width'], image_info['height'])}, "
                        f"got {img.size}"
                    )

            temp_path.replace(output_path)

            return {
                "success": True,
                "filename": filename,
                "final_split": job["final_split"],
                "original_split": original_split,
                "error": None
            }

        except Exception as e:

            if temp_path.exists():
                temp_path.unlink()

            if attempt == max_retries:
                return {
                    "success": False,
                    "filename": filename,
                    "final_split": job["final_split"],
                    "original_split": original_split,
                    "error": str(e)
                }

            time.sleep(2 * attempt)

# ==========================================
# RUN DOWNLOADS
# ==========================================

download_results = []

for index, job in enumerate(download_jobs, start=1):

    result = download_coco_image(job)
    download_results.append(result)

    if index % 25 == 0 or index == len(download_jobs):

        successful = sum(
            r["success"] for r in download_results
        )

        failed = len(download_results) - successful

        print(
            f"Progress: {index}/{len(download_jobs)} | "
            f"Downloaded: {successful} | Failed: {failed}"
        )

# ==========================================
# FINAL DOWNLOAD REPORT
# ==========================================

successful_downloads = [
    r for r in download_results if r["success"]
]

failed_downloads = [
    r for r in download_results if not r["success"]
]

print("\n" + "=" * 65)
print("DOWNLOAD SUMMARY")
print("=" * 65)

print("Expected images:", len(download_jobs))
print("Successfully downloaded:", len(successful_downloads))
print("Failed downloads:", len(failed_downloads))

for split in ("train", "valid"):

    count = sum(
        r["success"] and r["original_split"] == split
        for r in download_results
    )

    print(f"Original COCO {split}: {count} images")

if failed_downloads:

    print("\nFAILED IMAGES:")

    for item in failed_downloads:
        print(item["filename"], "->", item["error"])

    raise RuntimeError(
        "Some images failed to download. "
        "Do not proceed to conversion until all are downloaded."
    )

print("\nAll selected COCO images downloaded and verified!")
print("Location:", HYDRANT_IMAGE_DIR)


COCO FIRE HYDRANT IMAGE DOWNLOAD
Images to download: 364
Expected annotations: 400
Destination: /content/drive/MyDrive/Hazard_Detection/raw_datasets/obstacle/source_datasets/fire_hydrants/selected_images
Progress: 25/364 | Downloaded: 25 | Failed: 0
Progress: 50/364 | Downloaded: 50 | Failed: 0
Progress: 75/364 | Downloaded: 75 | Failed: 0
Progress: 100/364 | Downloaded: 100 | Failed: 0
Progress: 125/364 | Downloaded: 125 | Failed: 0
Progress: 150/364 | Downloaded: 150 | Failed: 0
Progress: 175/364 | Downloaded: 175 | Failed: 0
Progress: 200/364 | Downloaded: 200 | Failed: 0
Progress: 225/364 | Downloaded: 225 | Failed: 0
Progress: 250/364 | Downloaded: 250 | Failed: 0
Progress: 275/364 | Downloaded: 275 | Failed: 0
Progress: 300/364 | Downloaded: 300 | Failed: 0
Progress: 325/364 | Downloaded: 325 | Failed: 0
Progress: 350/364 | Downloaded: 350 | Failed: 0
Progress: 364/364 | Downloaded: 364 | Failed: 0

DOWNLOAD SUMMARY
Expected images: 364
Successfully downloaded: 364
Failed downloa